# Evasion-Aware Provenance–Forensics Fusion with Valid Operating Points
**Kaggle-ready research notebook** (free GPU tier: T4 / P100).

**Research gap.** No existing work provides one calibrated decision rule that fuses
(W) watermark/provenance evidence, (P) passive forensic evidence and (R) removal-trace evidence,
controls the false-positive rate on real images, and is stress-tested against adaptive attackers
and unseen generators / attacks.

**Pipeline.** `data -> canonicalise -> simulate provenance (watermark) -> attack suite -> branches W/P/R
-> fusion -> split-conformal p-values -> decision {authentic, provenance-AI, forensic-AI, evasion-suspected, abstain}`
-> metrics, thresholds, figures, LaTeX tables.

**Run order.** (1) set `PRESET` below, (2) *Run All*. Use `PRESET='smoke'` first (synthetic data, ~minutes, no GPU/internet needed),
then `'fast'` (~15-25 min on Kaggle) and finally `'standard'`.

Everything is seeded; all outputs go to `OUT_DIR` (CSV, JSON, LaTeX tables, PNG+PDF figures).

In [ ]:
# ============================================================================
# 0. IMPORTS, PRESETS, CONFIGURATION
# ============================================================================
import os, sys, io, json, time, math, random, platform, warnings, itertools
from pathlib import Path
from collections import deque
import numpy as np, pandas as pd
from PIL import Image
import scipy.fft as sfft
from scipy import ndimage as ndi
from scipy.optimize import minimize_scalar
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib
if os.environ.get('HEADLESS'): matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
warnings.filterwarnings('ignore')

# ---- choose one: 'smoke' (synthetic, minutes) | 'fast' | 'standard' -------------------------------------------------
PRESET = os.environ.get('PRESET', 'standard')

CFG = dict(
    DATA_ROOT=None,                 # None = auto-detect under /kaggle/input
    OUT_DIR='/kaggle/working/results' if os.path.isdir('/kaggle/working') else './results',
    SEED=2026,
    CROP=192,                       # native-resolution centre crop (multiple of 8; ArtiFact images are 200x200). CLIP input is resized to 224 inside the encoder
    CANON_QF=(75, 95),              # label-independent JPEG applied to *every* image (format-bias control)
    N_PER_FAKE_GEN=160, N_PER_REAL_SRC=450,
    HOLDOUT_FAKE_GENS=['stable_diffusion', 'stylegan3'],   # unseen generators (test only)
    HOLDOUT_REAL_SRC=['landscape'],                        # unseen real source (test only)
    SPLIT_FAKE=dict(train=.50, fusion=.20, cal=.00, test=.30),
    SPLIT_REAL=dict(train=.40, fusion=.10, cal=.25, test=.25),
    # provenance simulation (classical DCT spread-spectrum, 32-bit payload, luminance, key-seeded)
    WM_BITS=32, WM_ALPHA=2.0, WM_KEY=1337,
    # attack families seen during training; the rest are held out (unseen-attack generalisation)
    A_TRAIN=['jpeg', 'blur', 'rescale', 'noise'],
    V_TRAIN=5, V_FUSION=6, P_ADV_TRAIN=0.30,
    # threat-model priors used ONLY for fusion training (affect power, never validity)
    P_MARK_FAKE=0.50, P_FORGE_REAL=0.05, P_ADV_FAKE=0.50, P_ADV_REAL=0.10,
    ALPHAS=[0.05, 0.01], ALPHA_MAIN=0.01, TAU_ABSTAIN=0.20,
    CLIP_SOURCES=['/kaggle/input/clip-vit-base-patch16', 'openai/clip-vit-base-patch16'],
    BATCH=128, WORKERS=None,
    N_BOOT=500, N_SEEDS=5,
    # adaptive attacker / hardening ("self-play") / white-box PGD
    N_ADV=250, K_MAX=24, RUN_HARDENING=True, N_HARD_TR=600, N_HARD_FU=300, K_HARD=8,
    RUN_PGD=True, PGD_EPS=[2, 4, 8], PGD_STEPS=10, N_PGD=120,
    HARD_TARGET='E_WPR',                      # detector the hardening round mines hard examples against
    PSNR_FRONTIER=[None, 20, 25, 30, 35, 40], # quality-constrained attacker: minimum PSNR (dB) the attacked image must keep
)
PRESETS = {
    'smoke': dict(N_PER_FAKE_GEN=24, N_PER_REAL_SRC=100, N_BOOT=60, N_SEEDS=2, V_TRAIN=3, V_FUSION=3,
                  N_ADV=24, K_MAX=8, N_HARD_TR=40, N_HARD_FU=20, K_HARD=4, RUN_PGD=False,
                  ALPHAS=[0.10, 0.05], ALPHA_MAIN=0.10, OUT_DIR='./results_smoke'),
    'fast': dict(N_PER_FAKE_GEN=60, N_PER_REAL_SRC=150, N_BOOT=300, N_SEEDS=3, N_ADV=150, K_MAX=16,
                 N_HARD_TR=300, N_HARD_FU=150, K_HARD=6, ALPHAS=[0.10, 0.05], ALPHA_MAIN=0.05),
    'standard': dict(),
}
CFG.update(PRESETS[PRESET])
SMOKE = PRESET == 'smoke'
if SMOKE:   # synthetic dataset uses ArtiFact-style folder names
    CFG['HOLDOUT_FAKE_GENS'] = ['stable_diffusion', 'stylegan3']; CFG['HOLDOUT_REAL_SRC'] = ['landscape']

OUT = Path(CFG['OUT_DIR']); (OUT / 'figures').mkdir(parents=True, exist_ok=True); (OUT / 'tables').mkdir(exist_ok=True)
C = CFG['CROP']; NB = C // 8
ALPHA = CFG['ALPHA_MAIN']
T0 = time.time()
def log(*a): print(f'[{time.time()-T0:7.1f}s]', *a, flush=True)
def set_seed(s): random.seed(s); np.random.seed(s)
set_seed(CFG['SEED'])
log('preset =', PRESET, '| smoke =', SMOKE)

In [ ]:
# ============================================================================
# 1. ATTACK SUITE  ("laundering" adversary).  tier 0 = benign processing, tier 1 = adversarial
#    Threat model: attackers only apply tier-1 operations to AI images; real images mostly receive tier-0.
# ============================================================================
LEVELS = {
    'jpeg':    [(90, 0), (75, 0), (40, 1), (20, 1)],        # JPEG quality factor
    'blur':    [(0.5, 0), (1.5, 1), (2.5, 1)],              # Gaussian sigma (px)
    'noise':   [(2, 0), (10, 1), (20, 1)],                  # additive Gaussian sigma (0-255 scale)
    'rescale': [(0.9, 0), (0.5, 1), (0.35, 1)],             # down-then-up scale factor
    'bandsup': [(0.8, 0), (0.4, 1), (0.3, 1)],              # Butterworth low-pass cutoff / Nyquist
}
ALL_FAMS = list(LEVELS)
_f = np.fft.fftfreq(C); _RHO = np.sqrt(_f[None, :] ** 2 + _f[:, None] ** 2)

def _u8(a): return np.clip(np.rint(a), 0, 255).astype(np.uint8)
def atk_jpeg(x, q):
    b = io.BytesIO(); Image.fromarray(x).save(b, 'JPEG', quality=int(q)); b.seek(0)
    return np.array(Image.open(b).convert('RGB'))
def atk_blur(x, s): return _u8(ndi.gaussian_filter(x.astype(np.float32), sigma=(s, s, 0)))
def atk_noise(x, s, seed): return _u8(x + np.random.default_rng(seed).normal(0, s, x.shape))
def atk_rescale(x, s):
    n = max(24, int(round(C * s)))
    return np.array(Image.fromarray(x).resize((n, n), Image.Resampling.BICUBIC).resize((C, C), Image.Resampling.BICUBIC))
def atk_bandsup(x, r):
    M = 1.0 / (1.0 + (_RHO / (0.5 * r)) ** 8)
    X = np.fft.fft2(x.astype(np.float32), axes=(0, 1))
    return _u8(np.fft.ifft2(X * M[..., None], axes=(0, 1)).real)
def apply_chain(x, ops, seed):
    for j, (f, l, t) in enumerate(ops):
        if f == 'jpeg': x = atk_jpeg(x, l)
        elif f == 'blur': x = atk_blur(x, l)
        elif f == 'noise': x = atk_noise(x, l, seed + j)
        elif f == 'rescale': x = atk_rescale(x, l)
        elif f == 'bandsup': x = atk_bandsup(x, l)
    return x
def psnr(a, b):
    mse = float(np.mean((a.astype(np.float32) - b.astype(np.float32)) ** 2))
    return 99.0 if mse < 1e-9 else 10 * math.log10(255 ** 2 / mse)
def jitter_level(f, l, rng):
    if f == 'jpeg': return int(np.clip(l + rng.integers(-5, 6), 8, 98))
    v = l * (1 + rng.uniform(-0.15, 0.15))
    return float(min(v, 0.98)) if f in ('rescale', 'bandsup') else float(v)

In [ ]:
# ============================================================================
# 2. BRANCH W : simulated provider watermark  (DCT spread-spectrum, blind correlation detector)
#    Z_W ~ N(0,1) under H0 (unmarked image); we never rely on that: p-values are empirical (conformal).
# ============================================================================
_wrng = np.random.default_rng(CFG['WM_KEY'])
WM_COEF = [(1, 2), (2, 1), (2, 2), (0, 3), (3, 0), (1, 3), (3, 1), (2, 3)]
WM_U = np.array([u for u, v in WM_COEF]); WM_V = np.array([v for u, v in WM_COEF])
WM_PN = _wrng.choice([-1., 1.], size=(NB * NB, len(WM_COEF)))
WM_PAY = _wrng.choice([-1., 1.], size=CFG['WM_BITS'])
_perm = _wrng.permutation(NB * NB)
WM_G = np.zeros((CFG['WM_BITS'], NB * NB))
for _b, _ix in enumerate(np.array_split(_perm, CFG['WM_BITS'])): WM_G[_b, _ix] = 1.0
_bitof = (WM_G.T @ np.arange(CFG['WM_BITS'])).astype(int)
WM_DELTA = CFG['WM_ALPHA'] * WM_PAY[_bitof][:, None] * WM_PN
def _luma(x): return 0.299 * x[..., 0] + 0.587 * x[..., 1] + 0.114 * x[..., 2]
def _blocks(Y): return Y.reshape(NB, 8, NB, 8).transpose(0, 2, 1, 3).reshape(NB * NB, 8, 8)
def _unblocks(B): return B.reshape(NB, NB, 8, 8).transpose(0, 2, 1, 3).reshape(C, C)
def wm_embed(x, delta=None):
    d_ = WM_DELTA if delta is None else delta
    Y = _luma(x.astype(np.float32)); B = sfft.dctn(_blocks(Y), axes=(1, 2), norm='ortho')
    B[:, WM_U, WM_V] += d_
    d = (_unblocks(sfft.idctn(B, axes=(1, 2), norm='ortho')) - Y)[..., None]
    return _u8(x.astype(np.float32) + d)
def wm_decode(x):
    Y = _luma(x.astype(np.float32)); B = sfft.dctn(_blocks(Y), axes=(1, 2), norm='ortho')
    prod = B[:, WM_U, WM_V].astype(np.float64) * WM_PN
    S1 = prod.sum(1); S2 = (prod ** 2).sum(1)
    g1 = WM_G @ S1; g2 = WM_G @ S2; m = WM_G.sum(1) * prod.shape[1]
    mean = g1 / m; var = np.maximum((g2 - m * mean ** 2) / (m - 1), 1e-9)
    t = mean / np.sqrt(var / m)
    return float((WM_PAY * t).sum() / np.sqrt(len(t))), float((np.sign(mean) == WM_PAY).mean())

In [ ]:
# ============================================================================
# 3. BRANCH-R / P auxiliary: hand-crafted forensic descriptor (30-d)
#    radial log-power spectrum (24) + residual statistics (4) + JPEG-grid blockiness (2)
# ============================================================================
_NBIN = 24
_fy = np.fft.fftshift(np.fft.fftfreq(C)); _rho = np.sqrt(_fy[None, :] ** 2 + _fy[:, None] ** 2)
_RAD = np.minimum((_rho / 0.5 * _NBIN).astype(int), _NBIN)
_RADCNT = np.maximum(np.bincount(_RAD.ravel(), minlength=_NBIN + 1)[:_NBIN], 1)
_HANN = np.outer(np.hanning(C), np.hanning(C)).astype(np.float32)
def hf_features(x):
    g = _luma(x.astype(np.float32))
    F = np.fft.fftshift(np.fft.fft2((g - g.mean()) * _HANN)); P = F.real ** 2 + F.imag ** 2
    rad = np.bincount(_RAD.ravel(), weights=P.ravel(), minlength=_NBIN + 1)[:_NBIN] / _RADCNT
    r = g - ndi.gaussian_filter(g, 1.0)
    kurt = float(((r - r.mean()) ** 4).mean() / (r.var() ** 2 + 1e-8))
    dh = np.abs(np.diff(g, axis=1)); dv = np.abs(np.diff(g, axis=0))
    return np.concatenate([np.log10(rad + 1e-3),
                           [r.std(), kurt, np.abs(r).mean(), ndi.laplace(g).std(),
                            dh[:, 7::8].mean() - dh.mean(), dv[7::8, :].mean() - dv.mean()]]).astype(np.float32)

## Data
**Primary dataset (Kaggle):** `awsaf49/artifact-dataset` (ArtiFact: 25 generators, 8 real sources).
Attach it via *Add Input* on the Kaggle notebook page. The code auto-detects the folder layout and prints what it found.
Known issue handled here: JPEG/PNG and image-size shortcuts in real-vs-fake datasets
("Fake or JPEG?", arXiv 2403.17608). Every image is centre-cropped (no resampling) and re-compressed with a
*label-independent* random JPEG quality before any processing; a metadata-only leakage probe is reported.

In [ ]:
# ============================================================================
# 4. DATASET DISCOVERY, MANIFEST, SPLITS
# ============================================================================
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff')
# Folder names follow the ArtiFact release; family grouping is EDITABLE (verify against the ArtiFact paper).
ARTIFACT_REAL = ['afhq', 'celebahq', 'coco', 'ffhq', 'imagenet', 'landscape', 'lsun', 'metfaces']
FAKE_FAMILY = {
    'diffusion': ['ddpm', 'denoising_diffusion_gan', 'glide', 'latent_diffusion', 'palette', 'stable_diffusion', 'vq_diffusion'],
    'gan': ['big_gan', 'cips', 'cycle_gan', 'diffusion_gan', 'gansformer', 'gau_gan', 'pro_gan', 'projected_gan',
            'sfhq', 'star_gan', 'stylegan1', 'stylegan2', 'stylegan3'],
    'other': ['generative_inpainting', 'lama', 'mat', 'taming_transformer', 'face_synthetics'],
}
FAKE2FAM = {g: f for f, gs in FAKE_FAMILY.items() for g in gs}

def _synth_dataset(root, rng, n_real, n_fake):
    """Synthetic ArtiFact-like tree for the smoke test. Reals: JPEG; fakes: PNG (reproduces the format shortcut)."""
    def fractal(n, beta):
        f = np.fft.fftfreq(n); R = np.sqrt(f[None, :] ** 2 + f[:, None] ** 2); R[0, 0] = 1
        a = np.fft.ifft2((rng.normal(size=(n, n)) + 1j * rng.normal(size=(n, n))) / R ** (beta / 2)).real
        return (a - a.mean()) / (a.std() + 1e-9)
    def make(kind, n=272):
        base = fractal(n, rng.uniform(1.8, 2.4)); tint = rng.uniform(-.3, .3, 3)
        img = np.stack([base * .9 + .4 * fractal(n, 2.0) + t for t in tint], -1) * 45 + 120
        if kind == 'gan':
            yy, xx = np.mgrid[:n, :n]; img = img + 7.0 * (((yy + xx) % 2) * 2 - 1)[..., None]
        elif kind == 'diffusion':
            img = ndi.gaussian_filter(img, (1.3, 1.3, 0)) + rng.normal(0, 3.0, img.shape)
        elif kind == 'other':
            img = np.round(img / 20) * 20
        elif kind == 'real_sharp':
            img = img + rng.normal(0, 2.0, img.shape)
        return _u8(img)
    spec = {'coco': 'real', 'ffhq': 'real', 'landscape': 'real_sharp', 'stable_diffusion': 'diffusion', 'stylegan3': 'gan',
            'big_gan': 'gan', 'ddpm': 'diffusion', 'lama': 'other'}
    for name, kind in spec.items():
        d = Path(root) / name; d.mkdir(parents=True, exist_ok=True)
        isreal = kind.startswith('real')
        for i in range(n_real if isreal else n_fake):
            im = Image.fromarray(make(kind))
            if isreal: im.save(d / f'{i}.jpg', quality=int(rng.integers(70, 96)))
            else: im.save(d / f'{i}.png')

def find_dataset_root(cfg):
    known = set(ARTIFACT_REAL) | set(FAKE2FAM)
    def score(p):
        try:
            names = os.listdir(p)
            return len({n.lower() for n in names} & known) if len(names) < 3000 else 0
        except OSError: return 0
    cands = [cfg['DATA_ROOT']] if cfg['DATA_ROOT'] else []
    base = '/kaggle/input'
    def walk(p, depth):
        yield p
        if depth == 0: return
        try: kids = [os.path.join(p, k) for k in os.listdir(p)]
        except OSError: return
        if len(kids) > 3000 or score(p) >= 3: return
        for k in kids:
            if os.path.isdir(k): yield from walk(k, depth - 1)
    if os.path.isdir(base): cands += list(walk(base, 3))
    best = max(cands, key=score, default=None)
    if best is None or score(best) < 3:
        tree = os.listdir(base) if os.path.isdir(base) else 'no /kaggle/input'
        raise FileNotFoundError(f'ArtiFact folders not found. Add dataset "awsaf49/artifact-dataset" via Add Input. Saw: {tree}')
    return best

def list_images(d, limit):
    out, stack = [], [d]
    while stack and len(out) < limit:
        cur = stack.pop()
        try:
            with os.scandir(cur) as it:
                for e in it:
                    if e.is_dir(follow_symlinks=False): stack.append(e.path)
                    elif e.name.lower().endswith(IMG_EXT): out.append(e.path)
        except OSError: pass
    return out

LOAD_FAIL = {}
def load_one(args):
    path, uid = args
    try:
        im = Image.open(path); w, h = im.size
        if min(w, h) < C:
            LOAD_FAIL.setdefault('too_small', f'{w}x{h} < crop {C}'); return None
        im = im.convert('RGB'); l, t = (w - C) // 2, (h - C) // 2; im = im.crop((l, t, l + C, t + C))
        lo, hi = CFG['CANON_QF']; qf = int(np.random.default_rng([CFG['SEED'], uid]).integers(lo, hi + 1))
        b = io.BytesIO(); im.save(b, 'JPEG', quality=qf); b.seek(0)
        return np.array(Image.open(b).convert('RGB')), w, h, os.path.splitext(path)[1].lower()
    except Exception as e:
        LOAD_FAIL.setdefault('error', repr(e)); return None

def build_manifest(root, cfg, rng):
    groups = []
    for name in sorted(os.listdir(root)):
        if not os.path.isdir(os.path.join(root, name)): continue
        n = name.lower()
        if n in ARTIFACT_REAL: groups.append((name, 0, 'real', n))
        elif n in FAKE2FAM: groups.append((name, 1, FAKE2FAM[n], ''))
        else: log('  (ignored folder)', name)
    assert groups, 'no recognised group folders'
    rows, arrs, uid = [], [], 0
    for name, label, fam, src in groups:
        cap = cfg['N_PER_FAKE_GEN'] if label else cfg['N_PER_REAL_SRC']
        files = list_images(os.path.join(root, name), limit=cap * 400); rng.shuffle(files)
        got, pos, skipped = [], 0, 0
        while len(got) < cap and pos < len(files):
            batch = files[pos:pos + int((cap - len(got)) * 1.3) + 4]; pos += len(batch)
            with ThreadPoolExecutor(32) as ex:
                res = list(ex.map(load_one, [(p, uid + j) for j, p in enumerate(batch)]))
            uid += len(batch)
            for p, r in zip(batch, res):
                if r is None: skipped += 1; continue
                if len(got) < cap: got.append((p, r))
            if not got and skipped >= 200:
                raise RuntimeError(f'All {skipped} images tried in "{name}" were rejected. First failure reasons: {LOAD_FAIL}. '
                                   f'Check CFG["CROP"] (now {C}) against the real image size.')
        for p, (a, w, h, ext) in got:
            arrs.append(a)
            rows.append(dict(path=p, group=name.lower(), label=label, gen_family=fam, real_src=src, orig_w=w, orig_h=h, ext=ext))
        log(f'  {name:24s} label={label} family={fam:9s} kept={len(got):5d} skipped(small/corrupt)={skipped}')
    man = pd.DataFrame(rows)
    man['fmt'] = man.ext.map(lambda e: 'jpg' if e in ('.jpg', '.jpeg') else 'png' if e == '.png' else 'oth')
    mn = np.minimum(man.orig_w, man.orig_h)
    man['sizetier'] = np.where(mn < 256, 'xs', np.where(mn < 384, 's', np.where(mn < 768, 'm', 'l')))
    man['bin'] = man.fmt + '_' + man.sizetier           # observable at test time -> Mondrian conformal bins
    return man, np.stack(arrs)

def assign_splits(man, cfg, rng):
    avail_f = sorted(man[man.label == 1].group.unique()); avail_r = sorted(man[man.label == 0].group.unique())
    hf = [g for g in cfg['HOLDOUT_FAKE_GENS'] if g in avail_f]; hr = [g for g in cfg['HOLDOUT_REAL_SRC'] if g in avail_r]
    if len(hf) < 2:   # auto-pick: one diffusion + one gan if possible
        dif = [g for g in avail_f if FAKE2FAM.get(g) == 'diffusion']; gan = [g for g in avail_f if FAKE2FAM.get(g) == 'gan']
        hf = ([rng.choice(dif)] if dif else []) + ([rng.choice(gan)] if gan else [])
    if not hr and len(avail_r) > 1: hr = [rng.choice(avail_r)]
    man['hold'] = man.group.isin(hf + hr); man['split'] = ''
    for g, sub in man.groupby('group'):
        idx = sub.index.to_numpy().copy(); rng.shuffle(idx)
        if g in hf + hr: man.loc[idx, 'split'] = 'test'; continue
        ratios = cfg['SPLIT_FAKE'] if sub.label.iloc[0] == 1 else cfg['SPLIT_REAL']
        cuts = (np.cumsum([ratios[k] for k in ('train', 'fusion', 'cal', 'test')])[:-1] * len(idx)).astype(int)
        for k, part in zip(('train', 'fusion', 'cal', 'test'), np.split(idx, cuts)): man.loc[part, 'split'] = k
    log('hold-out unseen generators:', hf, '| unseen real source:', hr)
    return man, hf, hr

if SMOKE:
    root = './smoke_data'
    if not os.path.isdir(root): _synth_dataset(root, np.random.default_rng(0), CFG['N_PER_REAL_SRC'] + 8, CFG['N_PER_FAKE_GEN'] + 8)
else:
    root = find_dataset_root(CFG)
log('dataset root:', root)
rng = np.random.default_rng(CFG['SEED'])
MAN, HOLD_FAKE, HOLD_REAL = None, None, None
MAN, IMGS = build_manifest(root, CFG, rng)
MAN, HOLD_FAKE, HOLD_REAL = assign_splits(MAN, CFG, rng)
log('images in memory:', IMGS.shape, f'({IMGS.nbytes/1e9:.2f} GB)')
print(MAN.groupby(['split', 'label']).size().unstack(fill_value=0))

# metadata-only leakage probe: can file format / original size alone predict the label?
from sklearn.model_selection import cross_val_predict
_Xp = pd.get_dummies(MAN[['fmt', 'sizetier']]).assign(w=MAN.orig_w, h=MAN.orig_h).astype(float)
_pp = cross_val_predict(HistGradientBoostingClassifier(max_iter=60), _Xp, MAN.label, cv=5, method='predict_proba')[:, 1]
LEAK_AUC = roc_auc_score(MAN.label, _pp)
log(f'format/size leakage probe AUC = {LEAK_AUC:.3f}  ' + ('(!! HIGH: report it, rely on canonicalisation + held-out groups)' if LEAK_AUC > 0.75 else '(ok)'))

In [ ]:
# ============================================================================
# 5. VIEW SPECIFICATIONS  (image, watermark?, attack chain, seed)  and parallel rendering
# ============================================================================
def sample_ops(rng, fams, p_adv, jitter=True, p_none=0.4):
    if rng.random() < p_adv: pool = [(f, l, t) for f in fams for (l, t) in LEVELS[f] if t == 1]
    else:
        if rng.random() < p_none: return ()
        pool = [(f, l, t) for f in fams for (l, t) in LEVELS[f] if t == 0]
    f, l, t = pool[rng.integers(len(pool))]
    return ((f, jitter_level(f, l, rng) if jitter else l, t),)

def make_specs(img_ids, V, fams, p_adv, p_mark, rng, jitter=True, p_none=0.4):
    """p_adv, p_mark: dict label->prob. Sampling is independent of content except through the stated threat model."""
    lab = MAN.label.to_numpy(); out = []
    for i in img_ids:
        for _ in range(V):
            mark = int(rng.random() < p_mark[lab[i]])
            out.append((int(i), mark, sample_ops(rng, fams, p_adv[lab[i]], jitter, p_none), int(rng.integers(1 << 30))))
    return out

def test_specs(img_ids):
    ops_list = [()] + [((f, l, t),) for f in ALL_FAMS for (l, t) in LEVELS[f]]
    return [(int(i), m, ops, 7 * int(i) + 13) for i in img_ids for m in (0, 1) for ops in ops_list]

TIER1 = {f: [l for (l, t) in LEVELS[f] if t == 1] for f in ALL_FAMS}
def candidate_specs(img_ids, K, fams, rng, mark=1):
    """Adaptive-attacker candidates: random compositions of 1-2 tier-1 operations drawn from `fams`."""
    out = []
    for i in img_ids:
        for k in range(K):
            n = 1 if rng.random() < .5 else 2
            fs = rng.choice(fams, size=min(n, len(fams)), replace=False)
            ops = tuple((str(f), jitter_level(str(f), TIER1[str(f)][rng.integers(len(TIER1[str(f)]))], rng), 1) for f in fs)
            out.append((int(i), mark, ops, int(rng.integers(1 << 30))))
    return out

CONT_RANGE = {'jpeg': (15, 95), 'blur': (0.3, 3.0), 'noise': (1, 25), 'rescale': (0.3, 0.95), 'bandsup': (0.25, 0.9)}
def candidate_specs_cont(img_ids, K, fams, rng, mark=1):
    """Adaptive-attacker library with CONTINUOUS strengths (mild ... destructive), so attack success can be traded against image quality."""
    out = []
    for i in img_ids:
        for k in range(K):
            n = 1 if rng.random() < .5 else 2
            fs = [str(f) for f in rng.choice(fams, size=min(n, len(fams)), replace=False)]; ops = []
            for f in fs:
                lo, hi = CONT_RANGE[f]; v = rng.uniform(lo, hi)
                ops.append((f, int(round(v)) if f == 'jpeg' else float(v), 1))
            out.append((int(i), mark, tuple(ops), int(rng.integers(1 << 30))))
    return out

def specs_meta(specs):
    img = np.array([s[0] for s in specs]); m = MAN.iloc[img][['label', 'group', 'gen_family', 'real_src', 'split', 'hold', 'bin']].reset_index(drop=True)
    m['img'] = img; m['mark'] = [s[1] for s in specs]
    m['fam'] = ['+'.join(o[0] for o in s[2]) or 'none' for s in specs]
    m['level'] = [s[2][0][1] if s[2] else 0.0 for s in specs]
    m['tier'] = [max((o[2] for o in s[2]), default=0) for s in specs]
    m['nops'] = [len(s[2]) for s in specs]
    return m

def _work(chunk, want_psnr=False):
    n = len(chunk)
    imgs = np.empty((n, C, C, 3), np.uint8); zw = np.empty(n, np.float32); ba = np.empty(n, np.float32)
    hf = np.empty((n, _NBIN + 6), np.float32); ps = np.full(n, np.nan, np.float32)
    for i, (idx, mark, ops, seed) in enumerate(chunk):
        x0 = wm_embed(IMGS[idx]) if mark else IMGS[idx]
        x = apply_chain(x0, ops, seed); imgs[i] = x
        zw[i], ba[i] = wm_decode(x); hf[i] = hf_features(x)
        if want_psnr and ops: ps[i] = psnr(x0, x)
    return imgs, zw, ba, hf, ps

POOL = None
_nw = CFG['WORKERS'] or min(4, os.cpu_count() or 1)
if _nw > 1: POOL = ProcessPoolExecutor(max_workers=_nw)      # created BEFORE any CUDA/torch init (fork safety)
log('render workers:', _nw)

def _ordered(chunks, want_psnr):
    if POOL is None:
        for c in chunks: yield _work(c, want_psnr)
        return
    q = deque(); it = iter(chunks)
    for c in itertools.islice(it, 2 * _nw): q.append(POOL.submit(_work, c, want_psnr))
    while q:
        r = q.popleft().result(); nxt = next(it, None)
        if nxt is not None: q.append(POOL.submit(_work, nxt, want_psnr))
        yield r

def compute_views(specs, encoder, want_psnr=False, desc=''):
    B = CFG['BATCH']; chunks = [specs[i:i + B] for i in range(0, len(specs), B)]
    cl, zw, ba, hf, ps = [], [], [], [], []; t = time.time()
    for j, (imgs, z, b, h, p) in enumerate(_ordered(chunks, want_psnr)):
        cl.append(encoder.encode(imgs)); zw.append(z); ba.append(b); hf.append(h); ps.append(p)
        if (j + 1) % max(1, len(chunks) // 5) == 0 or j + 1 == len(chunks):
            log(f'  {desc}: {min((j+1)*B, len(specs))}/{len(specs)} views  ({time.time()-t:.0f}s)')
    return dict(meta=specs_meta(specs), clip=np.concatenate(cl), hf=np.concatenate(hf), zw=np.concatenate(zw),
                ba=np.concatenate(ba), psnr=np.concatenate(ps))

def featurize_images(imgs, encoder):
    """Features for already-rendered uint8 images (used for white-box PGD outputs)."""
    zb = [wm_decode(x) for x in imgs]
    return dict(clip=encoder.encode(imgs), zw=np.array([z for z, _ in zb], np.float32),
                ba=np.array([b for _, b in zb], np.float32), hf=np.stack([hf_features(x) for x in imgs]))

def vt_take(vt, sel):
    idx = np.flatnonzero(sel) if getattr(sel, 'dtype', None) == bool else np.asarray(sel)
    o = {k: vt[k][idx] for k in ('clip', 'hf', 'zw', 'ba', 'psnr') if k in vt}
    o['meta'] = vt['meta'].iloc[idx].reset_index(drop=True); return o
def vt_concat(vts):
    o = {k: np.concatenate([v[k] for v in vts]) for k in ('clip', 'hf', 'zw', 'ba', 'psnr')}
    o['meta'] = pd.concat([v['meta'] for v in vts], ignore_index=True); return o

In [ ]:
# ============================================================================
# 6. ENCODERS : frozen CLIP ViT-B/16 (GPU)  |  mock encoder (smoke test only)
# ============================================================================
class MockEncoder:
    name = 'mock-handcrafted'
    def encode(self, imgs): return np.stack([hf_features(x) for x in imgs]).astype(np.float32)

class ClipEncoder:
    def __init__(self, sources, bs=128):
        import torch
        from transformers import CLIPModel
        self.torch = torch; self.dev = 'cuda' if torch.cuda.is_available() else 'cpu'; self.bs = bs
        src = next((s for s in sources if os.path.exists(s)), sources[-1])
        log('loading CLIP from', src, 'on', self.dev)
        self.name = src
        self.model = CLIPModel.from_pretrained(src).to(self.dev).eval()
        for p in self.model.parameters(): p.requires_grad_(False)
        self.mean = torch.tensor([0.48145466, 0.4578275, 0.40821073], device=self.dev).view(1, 3, 1, 1)
        self.std = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=self.dev).view(1, 3, 1, 1)
    def forward_t(self, x):                      # x: float tensor in [0,1], N x 3 x 224 x 224 (differentiable)
        if x.shape[-1] != 224 or x.shape[-2] != 224:      # CLIP ViT-B/16 needs 224x224; resize is differentiable (PGD-safe)
            x = self.torch.nn.functional.interpolate(x, size=(224, 224), mode='bilinear', align_corners=False)
        x = (x - self.mean) / self.std
        pooled = self.model.vision_model(pixel_values=x).pooler_output
        e = self.model.visual_projection(pooled)
        return e / e.norm(dim=-1, keepdim=True)
    def encode(self, imgs):
        torch = self.torch; out = []
        with torch.no_grad():
            for i in range(0, len(imgs), self.bs):
                xb = torch.from_numpy(imgs[i:i + self.bs]).to(self.dev).permute(0, 3, 1, 2).float() / 255.
                with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=self.dev == 'cuda'):
                    e = self.forward_t(xb)
                out.append(e.float().cpu().numpy())
        return np.concatenate(out)

ENC = MockEncoder() if SMOKE else ClipEncoder(CFG['CLIP_SOURCES'])
log('encoder:', ENC.name)

## Experimental design (what is cached, and why)
Four **disjoint image splits**: *branch-train* (trains P and R), *fusion-train* (trains the fusion layer on frozen branches),
*calibration* (real images only; split-conformal), *test*. Two generators and one real source are held out entirely (test only),
and two attack families (`noise`, `bandsup`) are never seen in training.
* **Calibration views**: exactly **one** benign view per real image (views of one image are not exchangeable).
* **P / R training**: label-independent watermark and attack sampling, so neither branch can use "watermark present" as a shortcut.
* **Fusion training**: threat-model priors (attackers mostly launder AI images) - affects *power* only; validity comes from conformal calibration.

In [ ]:
# ============================================================================
# 7. RENDER + ENCODE ALL VIEW SETS ONCE (features are cached; everything after this is cheap CPU work)
# ============================================================================
rng = np.random.default_rng(CFG['SEED'] + 1)
ids = lambda sp: MAN.index[MAN.split == sp].to_numpy()
N_BENIGN = sum(t == 0 for f in ALL_FAMS for (l, t) in LEVELS[f])
cal_ids = ids('cal'); cal_ids = cal_ids[MAN.label.to_numpy()[cal_ids] == 0]
specs_tr = make_specs(ids('train'), CFG['V_TRAIN'], ALL_FAMS, {0: CFG['P_ADV_TRAIN'], 1: CFG['P_ADV_TRAIN']}, {0: .5, 1: .5}, rng)
specs_fu = make_specs(ids('fusion'), CFG['V_FUSION'], CFG['A_TRAIN'], {0: CFG['P_ADV_REAL'], 1: CFG['P_ADV_FAKE']},
                      {0: CFG['P_FORGE_REAL'], 1: CFG['P_MARK_FAKE']}, rng)
specs_cal = make_specs(cal_ids, 1, ALL_FAMS, {0: 0., 1: 0.}, {0: 0., 1: 0.}, rng, jitter=False, p_none=1 / (1 + N_BENIGN))
specs_te = test_specs(ids('test'))
log(f'views  train={len(specs_tr)}  fusion={len(specs_fu)}  cal={len(specs_cal)}  test={len(specs_te)}')
TR = compute_views(specs_tr, ENC, desc='train'); FU = compute_views(specs_fu, ENC, desc='fusion')
CAL = compute_views(specs_cal, ENC, desc='cal'); TE_VT = compute_views(specs_te, ENC, desc='test')

In [ ]:
# ============================================================================
# 8. MODELS: P (frozen embedding + logistic), R (gradient boosting), fusion, split-conformal
# ============================================================================
def _sig(z): return 1 / (1 + np.exp(-np.clip(z, -30, 30)))
def balanced_w(y):
    y = np.asarray(y); w = np.ones(len(y))
    for c in np.unique(y): w[y == c] = len(y) / (2.0 * (y == c).sum())
    return w
def fam_ok(meta, allowed):
    a = set(allowed) | {'none'}
    return meta['fam'].map(lambda s: all(c in a for c in s.split('+'))).to_numpy()
def fit_temperature(z, y):
    def nll(t): p = np.clip(_sig(z / t), 1e-6, 1 - 1e-6); return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
    return float(minimize_scalar(nll, bounds=(0.05, 20), method='bounded').x)

class Conformal:
    """Split-conformal p-values  p = (1 + #{cal >= s}) / (n + 1)  (super-uniform under exchangeability with real-benign data).
    * pooled  : one calibration set (all real sources).
    * minimax : p = max over per-source p-values.  Valid conditionally on every calibration source (p >= that source's own p-value);
                needs NO test-time covariate, and is conservative for sources outside the calibration pool."""
    def __init__(self, s, groups, min_group):
        self.g = np.sort(s)
        self.grp = [np.sort(s[groups == u]) for u in np.unique(groups) if (groups == u).sum() >= min_group]
    @staticmethod
    def _p(arr, s): return (1 + len(arr) - np.searchsorted(arr, s, side='left')) / (len(arr) + 1)
    def p(self, s, minimax=False):
        if minimax and len(self.grp) > 1: return np.max([self._p(a, s) for a in self.grp], axis=0)
        return self._p(self.g, s)

def xr(vt, pca): return np.c_[vt['hf'], pca.transform(vt['clip']), np.clip(vt['zw'], -10, 40), vt['ba']]
def fit_R(vt, seed):
    """Removal-trace detector. Label = adversarial-tier processing applied (independent of real/fake)."""
    X = vt['clip']; pca = PCA(n_components=min(32, X.shape[1]), random_state=seed).fit(X[::max(1, len(X) // 20000)])
    yR = (vt['meta'].tier.to_numpy() == 1).astype(int)
    clf = HistGradientBoostingClassifier(max_depth=5, learning_rate=.08, max_iter=250, early_stopping=True,
                                         validation_fraction=.15, random_state=seed)
    clf.fit(xr(vt, pca), yR, sample_weight=balanced_w(yR))
    return dict(pca=pca, clf=clf)

KEYS = ['W', 'P', 'R', 'F', 'F_WP', 'F_WR', 'F_PR', 'F_GBM', 'AVG', 'E_WP', 'E_WPR']
class Pipeline:
    def __init__(self, tr, fu, cal, seed=0, extra_tr=None, extra_fu=None, subsample=1.0):
        self.seed = seed; rng = np.random.default_rng(seed)
        trv = vt_take(tr, fam_ok(tr['meta'], CFG['A_TRAIN']))
        if extra_tr is not None: trv = vt_concat([trv, extra_tr])
        if subsample < 1:
            u = np.unique(trv['meta'].img); keep = rng.choice(u, int(subsample * len(u)), replace=False)
            trv = vt_take(trv, np.isin(trv['meta'].img.to_numpy(), keep))
        m = trv['meta']; X = trv['clip']; y = m.label.to_numpy(); img = m.img.to_numpy()
        # ---- P: standardise + L2 logistic on frozen embedding; C by inner validation AUC; temperature on validation
        u = np.unique(img); rng.shuffle(u); isval = np.isin(img, u[:max(2, int(.15 * len(u)))])
        sc = StandardScaler().fit(X[~isval]); best = None
        for Cc in (0.01, 0.1, 1.0):
            lr = LogisticRegression(C=Cc, max_iter=400, class_weight='balanced').fit(sc.transform(X[~isval]), y[~isval])
            auc = roc_auc_score(y[isval], lr.decision_function(sc.transform(X[isval])))
            if best is None or auc > best[0]: best = (auc, Cc, lr)
        zval = best[2].decision_function(sc.transform(X[isval]))
        self.P = dict(scaler=sc, lr=best[2], C=best[1], val_auc=best[0], T=fit_temperature(zval, y[isval]))
        # ---- R: removal-trace detector
        self.R = fit_R(trv, seed)
        # ---- fusion on frozen branches (separate images)
        fuv = vt_take(fu, fam_ok(fu['meta'], CFG['A_TRAIN']))
        if extra_fu is not None: fuv = vt_concat([fuv, extra_fu])
        b = self._base(fuv); F = np.c_[b['W'], b['P'], b['R']]; yf = fuv['meta'].label.to_numpy(); w = balanced_w(yf)
        self.fus = {}
        for name, cols in {'F': [0, 1, 2], 'F_WP': [0, 1], 'F_WR': [0, 2], 'F_PR': [1, 2]}.items():
            s2 = StandardScaler().fit(F[:, cols])
            self.fus[name] = (cols, s2, LogisticRegression(C=1.0, max_iter=500).fit(s2.transform(F[:, cols]), yf, sample_weight=w))
        self.gbm = HistGradientBoostingClassifier(max_depth=3, learning_rate=.1, max_iter=150, random_state=seed).fit(F, yf, sample_weight=w)
        # ---- calibration on real-benign views (one view per image)
        cb = self._base(cal); self.mu = {k: cb[k].mean() for k in 'WPR'}; self.sd = {k: cb[k].std() + 1e-9 for k in 'WPR'}
        self.cal_scores = self.scores(cal)
        mg = max(10, int(np.ceil(1 / min(CFG['ALPHAS']))) - 1)          # smallest group that can resolve alpha
        cgrp = cal['meta']['real_src'].to_numpy()
        self.cal = {k: Conformal(self.cal_scores[k], cgrp, mg) for k in KEYS}
    def _base(self, vt):
        return dict(W=np.clip(vt['zw'], -10, 40).astype(np.float64),
                    P=self.P['lr'].decision_function(self.P['scaler'].transform(vt['clip'])),
                    R=self.R['clf'].decision_function(xr(vt, self.R['pca'])))
    def scores(self, vt):
        b = self._base(vt); F = np.c_[b['W'], b['P'], b['R']]; s = dict(b)
        for name, (cols, s2, lr) in self.fus.items(): s[name] = lr.decision_function(s2.transform(F[:, cols]))
        s['F_GBM'] = self.gbm.decision_function(F)
        if hasattr(self, 'mu'):
            z = {k: (b[k] - self.mu[k]) / self.sd[k] for k in 'WPR'}; s['AVG'] = sum(z.values()) / 3
            # veto-resistant "evidence-sum": only POSITIVE evidence counts, so a branch that looks real can never cancel a verified watermark
            s['E_WP'] = np.maximum(z['W'], 0) + np.maximum(z['P'], 0); s['E_WPR'] = s['E_WP'] + np.maximum(z['R'], 0)
        return s
    def pvalues(self, scores, bins=None):
        """kind g = pooled calibration, kind x = minimax over real sources.  Bonferroni combinations are valid without independence."""
        P = {}
        for k in KEYS:
            P[f'p_{k}_g'] = self.cal[k].p(scores[k]); P[f'p_{k}_x'] = self.cal[k].p(scores[k], minimax=True)
        for kind in 'gx':
            P[f'p_B_WP_{kind}'] = np.minimum(1, 2 * np.minimum(P[f'p_W_{kind}'], P[f'p_P_{kind}']))
            P[f'p_BONF_{kind}'] = np.minimum(1, 3 * np.minimum.reduce([P[f'p_W_{kind}'], P[f'p_P_{kind}'], P[f'p_R_{kind}']]))
        return P

def tag_scenarios(m):
    adv = m.tier.to_numpy() == 1; real = m.label.to_numpy() == 0; mk = m.mark.to_numpy() == 1; clean = (m.fam.to_numpy() == 'none')
    s = np.full(len(m), 'other', dtype=object)
    s[real & ~adv & ~mk] = 'real_benign';          s[real & ~adv & mk] = 'real_forged_wm';  s[real & adv & ~mk] = 'real_strong_edit'; s[real & adv & mk] = 'real_forged_strong'
    s[~real & clean & mk] = 'fake_marked_clean';   s[~real & clean & ~mk] = 'fake_unmarked_clean'
    s[~real & ~adv & ~clean & mk] = 'fake_marked_benign';   s[~real & ~adv & ~clean & ~mk] = 'fake_unmarked_benign'
    s[~real & adv & mk] = 'fake_marked_laundered'; s[~real & adv & ~mk] = 'fake_unmarked_laundered'
    return s

def make_TE(pl, vt):
    m = vt['meta'].copy(); m['ba'] = vt['ba']; m['zw'] = vt['zw']; sc = pl.scores(vt); pv = pl.pvalues(sc)
    for k, v in sc.items(): m[f's_{k}'] = v
    m['s_BONF'] = -pv['p_BONF_g']; m['s_B_WP'] = -pv['p_B_WP_g']
    for k, v in pv.items(): m[k] = v
    m['scen'] = tag_scenarios(m); m['unseen'] = m['hold'].to_numpy()
    return m

log('fitting main pipeline (seed 0) ...')
PL = Pipeline(TR, FU, CAL, seed=0)
TE = make_TE(PL, TE_VT)
log(f"P inner-val AUC={PL.P['val_auc']:.3f} (C={PL.P['C']}), temperature T={PL.P['T']:.2f}; calibration reals n={len(CAL['meta'])}")
print(TE.scen.value_counts().to_string())

## Evaluation
All headline numbers are computed from **conformal decisions** (`p <= alpha`) with **cluster-bootstrap 95% CIs** that resample
*images* (not views), because views of one image are dependent. Scenarios:
`real_benign` (H0, the guaranteed FPR), `real_forged_wm` (watermark forged on a real image), `real_strong_edit` (stress FPR, *not* covered by the guarantee),
`fake_{marked,unmarked}_{clean,benign,laundered}`.

In [ ]:
# ============================================================================
# 9. STATISTICS HELPERS + IEEE-STYLE FIGURE SETUP
# ============================================================================
RNG_B = np.random.default_rng(CFG['SEED'] + 5); B_ = CFG['N_BOOT']
SCEN_ORDER = ['real_benign', 'real_forged_wm', 'real_strong_edit', 'real_forged_strong', 'fake_unmarked_clean', 'fake_marked_clean',
              'fake_unmarked_benign', 'fake_marked_benign', 'fake_unmarked_laundered', 'fake_marked_laundered']
# (display name, score key, calibration kind: g = pooled, x = minimax over real sources)
METH = [('W', 'W', 'g'), ('P', 'P', 'g'), ('R', 'R', 'g'), ('B_WP', 'B_WP', 'g'), ('BONF', 'BONF', 'g'), ('E_WP', 'E_WP', 'g'), ('E_WPR', 'E_WPR', 'g'),
        ('AVG', 'AVG', 'g'), ('F_GBM', 'F_GBM', 'g'), ('F', 'F', 'g'), ('E_WP-mm', 'E_WP', 'x')]
# NOTE v3: Bonferroni-minimax variants were removed: with ~100 calibration reals per source the smallest attainable p-value (x2 or x3 for
# Bonferroni) exceeds alpha=0.01, so they can never flag anything.  See table 'feasibility'.
MNAME = {'W': 'Watermark (W)', 'P': 'Passive (P)', 'R': 'Removal trace (R)', 'B_WP': 'Bonferroni(W,P)', 'BONF': 'Bonferroni(W,P,R)',
         'E_WP': 'Evidence-sum(W,P)', 'E_WPR': 'Evidence-sum(W,P,R)', 'AVG': 'Score average', 'F_GBM': 'Fusion GBM', 'F': 'Learned LR fusion (v1)',
         'B_WP-mm': 'Bonferroni(W,P) minimax', 'E_WP-mm': 'Evidence-sum(W,P) minimax', 'E_WPR-mm': 'Evidence-sum(W,P,R) minimax'}
COL = {'W': '#0072B2', 'P': '#D55E00', 'R': '#009E73', 'B_WP': '#CC79A7', 'BONF': '#882255', 'E_WP': '#E69F00', 'E_WPR': '#8C564B', 'AVG': '#7F7F7F',
       'F_GBM': '#56B4E9', 'F': '#000000', 'B_WP-mm': '#CC79A7', 'E_WP-mm': '#E69F00', 'E_WPR-mm': '#8C564B'}
LS = {'W': '-', 'P': '--', 'R': ':', 'B_WP': '-', 'BONF': '-.', 'E_WP': '-', 'E_WPR': '--', 'AVG': (0, (3, 1, 1, 1)), 'F_GBM': '--', 'F': ':',
      'B_WP-mm': (0, (5, 1)), 'E_WP-mm': (0, (5, 1)), 'E_WPR-mm': (0, (5, 1))}
MK = {'W': 'o', 'P': 's', 'R': '^', 'B_WP': 'v', 'BONF': 'v', 'E_WP': 'D', 'E_WPR': 'D', 'AVG': 'x', 'F_GBM': 'x', 'F': '*', 'B_WP-mm': 'P', 'E_WP-mm': 'P', 'E_WPR-mm': 'P'}
MAINS = ['E_WP', 'B_WP']          # recommended veto-resistant detectors (AI-origin evidence from W and P only)

plt.rcParams.update({'font.family': 'serif', 'font.serif': ['Times New Roman', 'DejaVu Serif'], 'mathtext.fontset': 'stix',
    'font.size': 8, 'axes.labelsize': 8, 'axes.titlesize': 8, 'legend.fontsize': 6.5, 'xtick.labelsize': 7, 'ytick.labelsize': 7,
    'figure.dpi': 110, 'savefig.dpi': 300, 'axes.linewidth': .6, 'lines.linewidth': 1.2, 'axes.grid': True, 'grid.alpha': .25,
    'grid.linewidth': .4, 'savefig.bbox': 'tight', 'figure.constrained_layout.use': True, 'pdf.fonttype': 42, 'ps.fonttype': 42, 'legend.frameon': False})
W1, W2 = 3.5, 7.16     # IEEE single / double column width (inches)
def savefig(fig, name):
    fig.savefig(OUT / 'figures' / f'{name}.png'); fig.savefig(OUT / 'figures' / f'{name}.pdf'); plt.show(); plt.close(fig)
def savetab(df, name, caption='', index=False, fmt='%.3f'):
    df.to_csv(OUT / 'tables' / f'{name}.csv', index=index)
    cols = list(df.reset_index().columns if index else df.columns); d = df.reset_index() if index else df
    esc = lambda t: str(t).replace('_', r'\_').replace('%', r'\%').replace('&', r'\&')
    body = ' \\\\\n'.join(' & '.join(esc(fmt % v if isinstance(v, (float, np.floating)) else v) for v in r) for r in d.itertuples(index=False))
    tex = (f'\\begin{{table}}[t]\\centering\\caption{{{esc(caption)}}}\\label{{tab:{name}}}\\small\n\\begin{{tabular}}{{{"l" * len(cols)}}}\\hline\n'
           f'{" & ".join(esc(c) for c in cols)} \\\\\\hline\n{body} \\\\\\hline\n\\end{{tabular}}\\end{{table}}\n')
    (OUT / 'tables' / f'{name}.tex').write_text(tex)

def flag(T, key, kind, a): return T[f'p_{key}_{kind}'].to_numpy() <= a
def _agg(T, mask, f):
    g = pd.DataFrame({'img': T['img'].to_numpy()[mask], 'f': np.asarray(f, float)[mask]}).groupby('img')['f'].agg(['sum', 'count'])
    return g['sum'].to_numpy(), g['count'].to_numpy()
def rate_ci(T, mask, f, B=None):
    """Mean rate over views with cluster (image-level) bootstrap 95% CI."""
    S, Cn = _agg(T, mask, f)
    if len(S) == 0: return np.nan, np.nan, np.nan
    est = S.sum() / Cn.sum(); idx = RNG_B.integers(0, len(S), (B or B_, len(S))); bs = S[idx].sum(1) / Cn[idx].sum(1)
    return est, np.quantile(bs, .025), np.quantile(bs, .975)
def diff_ci(T, mask, fa, fb, B=None):
    """Paired cluster bootstrap for rate(A) - rate(B) on the same images."""
    Sa, Ca = _agg(T, mask, fa); Sb, Cb = _agg(T, mask, fb)
    if len(Sa) == 0: return np.nan, np.nan, np.nan
    est = Sa.sum() / Ca.sum() - Sb.sum() / Cb.sum(); idx = RNG_B.integers(0, len(Sa), (B or B_, len(Sa)))
    bs = Sa[idx].sum(1) / Ca[idx].sum(1) - Sb[idx].sum(1) / Cb[idx].sum(1)
    return est, np.quantile(bs, .025), np.quantile(bs, .975)
def tpr_at_fpr(y, s, fpr):
    f, t, _ = roc_curve(y, s); return float(np.interp(fpr, f, t))
def pauc(y, s, mf=0.01):
    try: return float(roc_auc_score(y, s, max_fpr=mf))
    except Exception: return np.nan
def ece(y, p, w=None, bins=15):
    w = np.ones(len(y)) if w is None else w; edges = np.linspace(0, 1, bins + 1); tot = 0.
    for i in range(bins):
        m = (p > edges[i]) & (p <= edges[i + 1]) if i else (p >= 0) & (p <= edges[1])
        if w[m].sum() > 0: tot += w[m].sum() * abs(np.average(y[m], weights=w[m]) - np.average(p[m], weights=w[m]))
    return tot / w.sum()
SC = TE['scen'].to_numpy(); UN = TE['unseen'].to_numpy()

In [ ]:
# ============================================================================
# FIG 1 : system overview (for the paper's Method section)
# ============================================================================
fig, ax = plt.subplots(figsize=(W2, 2.5)); ax.set_xlim(0, 100); ax.set_ylim(0, 40); ax.axis('off'); ax.grid(False)
def box(x, y, w, h, txt, fc='#F2F2F2', ec='k', fs=6.5):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.25,rounding_size=0.8', fc=fc, ec=ec, lw=.7))
    ax.text(x + w / 2, y + h / 2, txt, ha='center', va='center', fontsize=fs)
def arr(x0, y0, x1, y1): ax.annotate('', xy=(x1, y1), xytext=(x0, y0), arrowprops=dict(arrowstyle='->', lw=.8))
box(0.5, 16, 12.5, 8, 'Input image $x$\n(centre crop,\nrandom JPEG)', '#FFFFFF', fs=6)
box(19, 29, 27, 8, 'W: DCT spread-spectrum\ndecoder $\\rightarrow Z_W$', '#DCEBF7', fs=6)
box(19, 16, 27, 8, 'P: frozen CLIP ViT-B/16\n+ logistic head $\\rightarrow s_P$', '#FBE3D6', fs=6)
box(19, 3, 27, 8, 'R: spectral + residual + grid\nfeatures, GBM $\\rightarrow s_R$', '#D6F0E6', fs=6)
box(52, 16, 14, 8, 'Fusion (LR)\n$s_F=f(Z_W,s_P,s_R)$', '#EEEEEE', fs=6)
box(70, 14, 16, 12, 'Split-conformal\n$p=\\frac{1+\\#\\{s_i\\geq s\\}}{n+1}$\n(real-benign\ncalibration)', '#FFF4CC', fs=6)
box(89.5, 4, 10, 32, 'Decision\n\nauthentic\nabstain\nprovenance-AI\nforensic-AI\nevasion-\nsuspected', '#FFFFFF', fs=5.5)
for yy in (33, 20, 7): arr(13.5, 20, 18.5, yy); arr(46.5, yy, 51.5, 20)
arr(66.5, 20, 69.5, 20); arr(86.5, 20, 89, 20)
ax.text(0.5, 38.2, 'Threat model: attacker launders AI images (tier-1 attacks) and may query the detector (adaptive)', fontsize=6, style='italic')
ax.text(0.5, 0.2, 'Guarantee: P(flag | real, benign) $\\leq \\alpha$ under exchangeability with the calibration reals', fontsize=6, style='italic')
savefig(fig, 'fig01_system_overview')

In [ ]:
# ============================================================================
# FIG 2 : watermark operating characteristics + strength sweep (documents the choice WM_ALPHA)
# ============================================================================
thrW = {a: np.quantile(PL.cal_scores['W'], 1 - a) for a in CFG['ALPHAS']}
fig, ax = plt.subplots(1, 2, figsize=(W2, 2.5))
mk_ = (SC == 'fake_marked_clean') | (SC == 'fake_marked_benign') | (SC == 'fake_marked_laundered')
for f in ALL_FAMS:
    xs, ys, lo, hi = [], [], [], []
    lv = sorted({l for l, t in LEVELS[f]}, reverse=f in ('jpeg', 'rescale', 'bandsup'))   # ordered mild -> strong
    for l in lv:
        msk = mk_ & (TE.fam.to_numpy() == f) & np.isclose(TE.level.to_numpy().astype(float), l)
        e, a_, b_ = rate_ci(TE, msk, flag(TE, 'W', 'g', ALPHA), 100); xs.append(l); ys.append(e); lo.append(a_); hi.append(b_)
    order = np.argsort(np.argsort(lv)); ax[0].errorbar(np.arange(len(lv)) + .06 * list(ALL_FAMS).index(f), ys, yerr=[np.array(ys) - lo, np.array(hi) - ys], marker='osv^D'[list(ALL_FAMS).index(f)],
                                                   ms=3, capsize=1.5, label=f)
ax[0].set_xticks(range(4)); ax[0].set_xticklabels(['L1\n(mildest)', 'L2', 'L3', 'L4\n(strongest)'])
ax[0].set_ylabel(f'W detection rate @ $\\alpha$={ALPHA}'); ax[0].set_title('(a) watermark survival per attack level'); ax[0].legend(ncol=2); ax[0].set_ylim(-.03, 1.05)
# strength sweep on training-split reals only (not used in any reported test metric)
sw_ids = MAN.index[(MAN.split == 'train') & (MAN.label == 0)].to_numpy()[:(30 if SMOKE else 100)]
alphas_sw = [1.0, 1.5, 2.0, 3.0, 4.0]; curves = {'clean': [], 'benign tier': [], 'adversarial tier': []}
null_z = np.array([wm_decode(IMGS[i])[0] for i in sw_ids]); thr_sw = np.quantile(null_z, .99)
for a_ in alphas_sw:
    dl = a_ * WM_PAY[_bitof][:, None] * WM_PN; dr = {'clean': [], 'benign tier': [], 'adversarial tier': []}
    for i in sw_ids:
        x0 = wm_embed(IMGS[i], dl); dr['clean'].append(wm_decode(x0)[0] > thr_sw)
        for f in ALL_FAMS:
            for l, t in LEVELS[f]:
                dr['benign tier' if t == 0 else 'adversarial tier'].append(wm_decode(apply_chain(x0, ((f, l, t),), 3))[0] > thr_sw)
    for k in curves: curves[k].append(np.mean(dr[k]))
for k, ls in zip(curves, ['-o', '--s', ':^']): ax[1].plot(alphas_sw, curves[k], ls, ms=3, label=k)
ax[1].axvline(CFG['WM_ALPHA'], color='gray', lw=.6, ls=':'); ax[1].set_xlabel('embedding strength $a$ (DCT units)')
ax[1].set_ylabel('detection rate @ null 99th pct'); ax[1].set_title('(b) strength sweep (train-split reals)'); ax[1].legend(); ax[1].set_ylim(-.03, 1.05)
savefig(fig, 'fig02_watermark_characterisation')
psnr_wm = float(np.mean([psnr(IMGS[i], wm_embed(IMGS[i])) for i in sw_ids[:30]])); log(f'embedding PSNR = {psnr_wm:.1f} dB (a={CFG["WM_ALPHA"]})')

In [ ]:
# ============================================================================
# FIG 3 : empirical null vs theory.  Threshold of W: conformal quantile vs N(0,1) vs Binomial(32,1/2) bit-accuracy
# ============================================================================
from math import comb
fig, ax = plt.subplots(1, 2, figsize=(W2, 2.4))
zc = PL.cal_scores['W']; ax[0].hist(zc, bins=30, density=True, color='#BBBBBB', ec='k', lw=.3, label='calibration reals')
xx = np.linspace(-4, 4, 200); ax[0].plot(xx, np.exp(-xx ** 2 / 2) / np.sqrt(2 * np.pi), 'k--', lw=.8, label='$N(0,1)$')
for a_, c_ in zip(CFG['ALPHAS'], ['#D55E00', '#0072B2']):
    ax[0].axvline(np.quantile(zc, 1 - a_), color=c_, lw=1, label=f'conformal thr. $\\alpha$={a_}')
ax[0].set_xlim(-4, 6); ax[0].set_xlabel('$Z_W$ on unmarked real images'); ax[0].set_ylabel('density'); ax[0].set_title('(a) empirical null of $Z_W$'); ax[0].legend()
nb = CFG['WM_BITS']; pmf = np.array([comb(nb, k) for k in range(nb + 1)]) / 2 ** nb
bm = TE[(TE.scen == 'real_benign') & (TE.mark == 0)]['ba'].to_numpy() * nb
ax[1].hist(bm, bins=np.arange(-.5, nb + 1.5), density=True, color='#BBBBBB', ec='k', lw=.3, label='unmarked real images')
ax[1].plot(range(nb + 1), pmf, 'k.-', lw=.8, ms=3, label='Binomial$(32,\\frac{1}{2})$')
for a_, c_ in zip(CFG['ALPHAS'], ['#D55E00', '#0072B2']):
    kk = next(k for k in range(nb + 1) if sum(pmf[k:]) <= a_); ax[1].axvline(kk - .5, color=c_, lw=1, ls='--', label=f'binomial thr. $\\alpha$={a_}: $\\geq${kk}/{nb}')
ax[1].set_xlabel('correctly decoded bits'); ax[1].set_title('(b) hard-decision bit accuracy under H0'); ax[1].legend()
savefig(fig, 'fig03_null_and_thresholds')
THR = pd.DataFrame([dict(alpha=a_, Z_W_conformal=np.quantile(zc, 1 - a_), Z_W_gaussian=float(__import__('scipy.stats', fromlist=['norm']).norm.ppf(1 - a_)),
                         binom_bits=next(k for k in range(nb + 1) if sum(pmf[k:]) <= a_),
                         P_threshold=np.quantile(PL.cal_scores['P'], 1 - a_), R_threshold=np.quantile(PL.cal_scores['R'], 1 - a_),
                         F_threshold=np.quantile(PL.cal_scores['F'], 1 - a_)) for a_ in CFG['ALPHAS']])
savetab(THR, 'thresholds', 'Operating thresholds derived from the calibration reals (global conformal).'); print(THR.round(3).to_string(index=False))

In [ ]:
# ============================================================================
# 10. MAIN RESULTS : rates with cluster-bootstrap CIs, AUC / partial AUC, and FIG 4-6
# ============================================================================
rows = []
for a_ in CFG['ALPHAS']:
    for (name, key, kind) in METH:
        fl = flag(TE, key, kind, a_)
        for sc_ in SCEN_ORDER:
            for grp, extra in (('all', np.ones(len(TE), bool)), ('unseen', UN), ('seen', ~UN)):
                msk = (SC == sc_) & extra
                if msk.sum() == 0: continue
                e, lo, hi = rate_ci(TE, msk, fl); rows.append(dict(alpha=a_, method=name, scenario=sc_, group=grp, rate=e, lo=lo, hi=hi, n_img=TE.img[msk].nunique()))
RES = pd.DataFrame(rows); savetab(RES, 'main_results_rates', 'Flag rate per scenario (FPR for real_*, TPR for fake_*) with 95% cluster-bootstrap CI.')
piv = RES[(RES.alpha == ALPHA) & (RES.group == 'all')].pivot(index='method', columns='scenario', values='rate').reindex([m[0] for m in METH])[SCEN_ORDER]
print(f'\nFlag rate at alpha={ALPHA} (all test views)'); print(piv.round(3).to_string())

neg = (SC == 'real_benign')
def auc_of(pos_mask, key):
    m = pos_mask | neg; y = pos_mask[m].astype(int); s = TE[f's_{key}'].to_numpy()[m]; return roc_auc_score(y, s), pauc(y, s)
arows = []
for sc_ in ['fake_unmarked_clean', 'fake_marked_clean', 'fake_unmarked_benign', 'fake_unmarked_laundered', 'fake_marked_laundered']:
    for grp, extra in (('seen', ~UN), ('unseen', UN)):
        pm = (SC == sc_) & extra
        if pm.sum() < 5: continue
        for key in ['W', 'P', 'R', 'B_WP', 'E_WP', 'E_WPR', 'F']:
            a1, a2 = auc_of(pm, key); arows.append(dict(scenario=sc_, generators=grp, method=MNAME[key], AUC=a1, pAUC_FPR1pct=a2))
AUCT = pd.DataFrame(arows); savetab(AUCT, 'auc_table', 'AUC and standardised partial AUC (FPR<=1%) against real_benign.')

# ---- FIG 4: ROC (log-FPR)
fig, axs = plt.subplots(1, 4, figsize=(W2, 2.0), sharey=True)
panels = [('(a) unmarked, clean, seen gen.', (SC == 'fake_unmarked_clean') & ~UN), ('(b) unmarked, clean, unseen gen.', (SC == 'fake_unmarked_clean') & UN),
          ('(c) marked, laundered', SC == 'fake_marked_laundered'), ('(d) unmarked, laundered', SC == 'fake_unmarked_laundered')]
for ax_, (ttl, pm) in zip(axs, panels):
    if pm.sum() < 5: ax_.set_title(ttl + '\n(n/a)'); continue
    m = pm | neg; y = pm[m].astype(int)
    for key in ['W', 'P', 'R', 'B_WP', 'E_WP', 'F']:
        f_, t_, _ = roc_curve(y, TE[f's_{key}'].to_numpy()[m]); ax_.plot(np.maximum(f_, 1e-4), t_, color=COL[key], ls=LS[key], label=MNAME[key], lw=1.5 if key in MAINS else .9)
    ax_.axvline(ALPHA, color='gray', lw=.5, ls=':'); ax_.set_xscale('log'); ax_.set_xlim(1e-3, 1); ax_.set_title(ttl, fontsize=6.5); ax_.set_xlabel('FPR (real, benign)')
axs[0].set_ylabel('TPR'); axs[0].legend(loc='lower right', fontsize=5.5); savefig(fig, 'fig04_roc')

# ---- FIG 5: conformal validity (nominal vs empirical FPR) and per-source FPR (pooled vs minimax calibration)
grid = [g for g in [0.005, 0.01, 0.02, 0.05, 0.1, 0.2] if g >= 1.0 / (len(CAL['meta']) + 1)]
fig, ax = plt.subplots(1, 2, figsize=(W2, 2.6)); ind = (SC == 'real_benign') & ~UN
for (name, key, kind) in [m for m in METH if m[0] in ('W', 'P', 'B_WP', 'E_WP', 'E_WP-mm', 'F')]:
    est = [rate_ci(TE, ind, flag(TE, key, kind, g), 100) for g in grid]
    ax[0].errorbar(grid, [e[0] for e in est], yerr=[[e[0] - e[1] for e in est], [e[2] - e[0] for e in est]], color=COL[name], ls=LS[name], marker=MK[name], ms=3, capsize=1.5, label=MNAME[name])
ax[0].plot([0, .2], [0, .2], 'k:', lw=.8, label='nominal'); ax[0].set_xlabel('nominal $\\alpha$'); ax[0].set_ylabel('empirical FPR (in-distribution reals)')
ax[0].set_title('(a) conformal validity'); ax[0].legend(fontsize=5.5)
srcs = sorted(TE[SC == 'real_benign'].real_src.unique()); a2 = max(ALPHA, 0.05) if ALPHA < 0.05 else ALPHA
bar_m = [m for m in METH if m[0] in ('P', 'E_WP', 'E_WP-mm')]; wd = .8 / len(bar_m)
for j, (name, key, kind) in enumerate(bar_m):
    vals = [rate_ci(TE, (SC == 'real_benign') & (TE.real_src.to_numpy() == s_), flag(TE, key, kind, a2), 100) for s_ in srcs]
    ax[1].bar(np.arange(len(srcs)) + (j - (len(bar_m) - 1) / 2) * wd, [v[0] for v in vals], wd, yerr=[[v[0] - v[1] for v in vals], [v[2] - v[0] for v in vals]],
              capsize=1, color=COL[name], ec='k', lw=.3, hatch='//' if kind == 'x' else None, label=MNAME[name], error_kw=dict(lw=.5))
ax[1].axhline(a2, color='k', ls=':', lw=.8); ax[1].set_xticks(range(len(srcs))); ax[1].set_xticklabels([s_ + ('*' if s_ in HOLD_REAL else '') for s_ in srcs], rotation=45, ha='right')
ax[1].set_ylabel(f'FPR @ $\\alpha$={a2}'); ax[1].set_title('(b) per real source (* = never in calibration)'); ax[1].legend(fontsize=5.5)
savefig(fig, 'fig05_conformal_validity')
src_rows = []
for (name, key, kind) in METH:
    for s_ in srcs:
        msk = (SC == 'real_benign') & (TE.real_src.to_numpy() == s_); e = rate_ci(TE, msk, flag(TE, key, kind, ALPHA), 100)
        src_rows.append(dict(method=name, source=s_, never_in_calibration=s_ in HOLD_REAL, FPR=e[0], lo=e[1], hi=e[2], n_img=TE.img[msk].nunique()))
SRCT = pd.DataFrame(src_rows); savetab(SRCT, 'fpr_by_source', f'Per-real-source FPR at alpha={ALPHA} (pooled vs minimax calibration).')
WORST = SRCT.groupby('method').FPR.max().reindex([m[0] for m in METH]); print(f'\nWorst-source FPR at alpha={ALPHA}:'); print(WORST.round(3).to_string())

# ---- feasibility: smallest attainable p-value per method (a detector whose minimum p exceeds alpha can never flag)
_n_pool = len(PL.cal['W'].g); _n_grp = [len(x) for x in PL.cal['W'].grp]; _n_min = min(_n_grp) if len(_n_grp) > 1 else _n_pool; fe = []
for (name, key, kind) in METH:
    f_ = 2 if key == 'B_WP' else 3 if key == 'BONF' else 1
    pm_ = f_ * (1.0 / (_n_min + 1) if kind == 'x' else 1.0 / (_n_pool + 1))
    for a_ in CFG['ALPHAS']: fe.append(dict(method=name, alpha=a_, min_attainable_p=pm_, feasible=bool(pm_ <= a_)))
FEAS = pd.DataFrame(fe); savetab(FEAS, 'feasibility', f'Smallest attainable p-value (pooled n={_n_pool}, smallest source n={_n_min}); infeasible = can never flag at that alpha.')
print('\nInfeasible (method, alpha):', [(r.method, r.alpha) for r in FEAS.itertuples() if not r.feasible] or 'none')

# ---- FIG 6: heat-map of flag rates
fig, ax = plt.subplots(figsize=(W2, 3.4)); data = piv.values.astype(float)
im = ax.imshow(data, cmap='viridis', vmin=0, vmax=1, aspect='auto')
for i in range(data.shape[0]):
    for j in range(data.shape[1]): ax.text(j, i, f'{data[i, j]:.2f}', ha='center', va='center', fontsize=6, color='w' if data[i, j] < .6 else 'k')
ax.set_xticks(range(len(SCEN_ORDER))); ax.set_xticklabels([s.replace('_', ' ') for s in SCEN_ORDER], fontsize=5.2, rotation=35, ha='right'); ax.set_yticks(range(len(piv))); ax.set_yticklabels([MNAME[m] for m in piv.index], fontsize=6.5)
ax.axvline(3.5, color='w', lw=2); ax.grid(False); ax.set_title(f'Flag rate at $\\alpha$={ALPHA} (left of white bar: FPR, lower is better; right: TPR, higher is better)', fontsize=7)
fig.colorbar(im, ax=ax, fraction=.02); savefig(fig, 'fig06_flag_rate_heatmap')

In [ ]:
# ============================================================================
# 11. EVASION ANALYSIS (Evennou-style compound success, extended to W + P + R + fusion)   -> FIG 7
#     "success" for the attacker = the AI image is NOT flagged.  Compound = evades W, P and R simultaneously.
# ============================================================================
def evasion_table(T, scen, alpha):
    rows = []
    for f in ALL_FAMS:
        msk = (T.scen.to_numpy() == scen) & (T.fam.to_numpy() == f)
        if msk.sum() == 0: continue
        nf = {k: ~flag(T, k, 'g', alpha) for k in ('W', 'P', 'R', 'F', 'B_WP', 'E_WP', 'E_WPR')}
        out = dict(family=f, unseen_family=f not in CFG['A_TRAIN'])
        for name, fl in (('W', nf['W']), ('P', nf['P']), ('R', nf['R']), ('compound(W,P,R)', nf['W'] & nf['P'] & nf['R']),
                         ('LR fusion (v1)', nf['F']), ('Bonferroni(W,P)', nf['B_WP']), ('Evidence-sum(W,P)', nf['E_WP'])):
            out[name], out[name + '_lo'], out[name + '_hi'] = rate_ci(T, msk, fl, 150)
        rows.append(out)
    return pd.DataFrame(rows)
EVN = ['W', 'P', 'R', 'compound(W,P,R)', 'LR fusion (v1)', 'Bonferroni(W,P)', 'Evidence-sum(W,P)']
EVC = {'W': COL['W'], 'P': COL['P'], 'R': COL['R'], 'compound(W,P,R)': '#BBBBBB', 'LR fusion (v1)': '#000000', 'Bonferroni(W,P)': COL['B_WP'], 'Evidence-sum(W,P)': COL['E_WP']}
EV = evasion_table(TE, 'fake_marked_laundered', ALPHA); EVU = evasion_table(TE, 'fake_unmarked_laundered', ALPHA)
savetab(EV[['family', 'unseen_family'] + EVN], 'evasion_marked', f'Attack success (AI image NOT flagged) at alpha={ALPHA}, provider-marked images.')
savetab(EVU[['family', 'unseen_family'] + EVN], 'evasion_unmarked', f'Attack success at alpha={ALPHA}, unmarked AI images.')
fig, axs = plt.subplots(1, 2, figsize=(W2, 2.6), sharey=True)
for ax_, tab, ttl in zip(axs, (EV, EVU), ('(a) provider-marked AI images', '(b) unmarked AI images')):
    w = .12
    for j, nme in enumerate(EVN):
        ax_.bar(np.arange(len(tab)) + (j - 3) * w, tab[nme], w, yerr=[tab[nme] - tab[nme + '_lo'], tab[nme + '_hi'] - tab[nme]], color=EVC[nme], ec='k', lw=.3,
                capsize=1, error_kw=dict(ecolor='0.35', lw=.5), label=nme if ax_ is axs[0] else None, hatch='//' if nme == 'compound(W,P,R)' else None)
    ax_.set_xticks(range(len(tab))); ax_.set_xticklabels([f + ('*' if u else '') for f, u in zip(tab.family, tab.unseen_family)]); ax_.set_title(ttl); ax_.set_ylim(0, 1.05)
axs[0].set_ylabel(f'attack success @ $\\alpha$={ALPHA} (lower = better)')
_h, _l = axs[0].get_legend_handles_labels(); fig.legend(_h, _l, loc='upper center', ncol=4, fontsize=6, bbox_to_anchor=(0.5, 1.12))
fig.supxlabel('attack family (tier-1 levels pooled);  * = family never seen in training', fontsize=7); savefig(fig, 'fig07_evasion_by_family')
print('\nAttack success, marked images:'); print(EV[['family', 'unseen_family'] + EVN].round(3).to_string(index=False))

In [ ]:
# ============================================================================
# 12. GENERALISATION: leave-one-generator-FAMILY-out (P) and leave-one-attack-FAMILY-out (R)   -> FIG 8
# ============================================================================
trv = vt_take(TR, fam_ok(TR['meta'], CFG['A_TRAIN'])); mtr = trv['meta']
clean_neg = (SC == 'real_benign')
lg_rows = []
for fam_ in sorted(MAN[MAN.label == 1].gen_family.unique()):
    keep = ~((mtr.label.to_numpy() == 1) & (mtr.gen_family.to_numpy() == fam_))
    if keep.sum() == len(keep) or (mtr.label.to_numpy()[keep] == 1).sum() < 10: continue
    sc_ = StandardScaler().fit(trv['clip'][keep]); lr_ = LogisticRegression(C=PL.P['C'], max_iter=400, class_weight='balanced').fit(sc_.transform(trv['clip'][keep]), mtr.label.to_numpy()[keep])
    pos = (SC == 'fake_unmarked_clean') & (TE.gen_family.to_numpy() == fam_)
    if pos.sum() < 5: continue
    m = pos | clean_neg; y = pos[m].astype(int); s = lr_.decision_function(sc_.transform(TE_VT['clip'][m]))
    lg_rows.append(dict(held_out_family=fam_, n_pos_views=int(pos.sum()), AUC=roc_auc_score(y, s), TPR_at_FPR1pct=tpr_at_fpr(y, s, .01), TPR_at_FPR5pct=tpr_at_fpr(y, s, .05)))
LOGO = pd.DataFrame(lg_rows); savetab(LOGO, 'logo_passive', 'Leave-one-generator-family-out: passive branch P.')
la_rows = []
adv_pos_all = (TE.tier.to_numpy() == 1)
for f in ALL_FAMS:
    sel = (mtr.fam.to_numpy() == f) | np.array([f in s.split('+') for s in mtr.fam])
    Rm = fit_R(vt_take(TR, ~sel), 0)                                    # R trained WITHOUT family f (benign or adversarial levels)
    te_pos = (TE.fam.to_numpy() == f) & adv_pos_all
    te_neg = (TE.tier.to_numpy() == 0) & ~np.array([f in s.split('+') for s in TE.fam])
    m = te_pos | te_neg; y = te_pos[m].astype(int); s = Rm['clf'].decision_function(xr(vt_take(TE_VT, m), Rm['pca']))
    s_main = TE['s_R'].to_numpy()[m]
    la_rows.append(dict(attack_family=f, LOAO_AUC=roc_auc_score(y, s), main_R_AUC=roc_auc_score(y, s_main), main_R_saw_family=f in CFG['A_TRAIN']))
LOAO = pd.DataFrame(la_rows); savetab(LOAO, 'loao_removal', 'Leave-one-attack-family-out AUC of the removal-trace branch R (adversarial vs benign/none).')
fig, ax = plt.subplots(1, 2, figsize=(W2, 2.3))
ax[0].bar(range(len(LOGO)), LOGO.AUC, color='#D55E00', ec='k', lw=.4, label='AUC'); ax[0].bar(np.arange(len(LOGO)) + .0, LOGO.TPR_at_FPR1pct, .4, color='#999999', ec='k', lw=.4, label='TPR@FPR=1%')
ax[0].set_xticks(range(len(LOGO))); ax[0].set_xticklabels(LOGO.held_out_family); ax[0].set_ylim(0, 1.05); ax[0].set_title('(a) P: leave-one-generator-family-out'); ax[0].legend()
xi = np.arange(len(LOAO)); ax[1].bar(xi - .18, LOAO.LOAO_AUC, .36, color='#009E73', ec='k', lw=.4, label='R retrained without family'); ax[1].bar(xi + .18, LOAO.main_R_AUC, .36, color='#999999', ec='k', lw=.4, label='main R')
ax[1].set_xticks(xi); ax[1].set_xticklabels([f + ('' if s_ else '*') for f, s_ in zip(LOAO.attack_family, LOAO.main_R_saw_family)]); ax[1].set_ylim(0, 1.05)
ax[1].set_title('(b) R: leave-one-attack-family-out (* unseen by main R)'); ax[1].legend(); savefig(fig, 'fig08_generalisation')
print('\nLOGO (passive P):'); print(LOGO.round(3).to_string(index=False)); print('\nLOAO (removal R):'); print(LOAO.round(3).to_string(index=False))

In [ ]:
# ============================================================================
# 13. CALIBRATION OF P (why raw probabilities cannot be trusted OOD) + RISK-COVERAGE      -> FIG 9, FIG 10
# ============================================================================
cl = ((SC == 'fake_unmarked_clean') | ((SC == 'real_benign') & (TE.fam.to_numpy() == 'none'))) & (TE.mark.to_numpy() == 0)
fig, axs = plt.subplots(1, 2, figsize=(W2, 2.5)); cal_rows = []
for ax_, (nm, grp) in zip(axs, (('seen generators / sources', ~UN), ('unseen generators / source', UN))):
    m = cl & grp
    if m.sum() < 20: ax_.set_title(nm + ' (n/a)'); continue
    y = (SC[m] != 'real_benign').astype(int); sP = TE['s_P'].to_numpy()[m]; w = np.where(y == 1, .5 / max(y.sum(), 1), .5 / max((1 - y).sum(), 1))
    for lab, p, c_, ls_ in (('raw logistic', _sig(sP), '#D55E00', '--'), ('temperature-scaled', _sig(sP / PL.P['T']), '#0072B2', '-')):
        e = ece(y, p, w); br = float(np.average((p - y) ** 2, weights=w)); nll = float(-np.average(y * np.log(np.clip(p, 1e-6, 1)) + (1 - y) * np.log(np.clip(1 - p, 1e-6, 1)), weights=w))
        edges = np.linspace(0, 1, 11); xs, ys = [], []
        for i in range(10):
            mm = (p >= edges[i]) & (p < edges[i + 1] if i < 9 else p <= 1)
            if w[mm].sum() > 0: xs.append(np.average(p[mm], weights=w[mm])); ys.append(np.average(y[mm], weights=w[mm]))
        ax_.plot(xs, ys, ls_, marker='o', ms=3, color=c_, label=f'{lab} (ECE={e:.3f})'); cal_rows.append(dict(group=nm, probs=lab, ECE=e, Brier=br, NLL=nll))
    ax_.plot([0, 1], [0, 1], 'k:', lw=.8); ax_.set_xlabel('predicted P(AI)'); ax_.set_ylabel('observed frequency'); ax_.set_title(nm); ax_.legend(loc='upper left')
savefig(fig, 'fig09_reliability'); CALT = pd.DataFrame(cal_rows); savetab(CALT, 'calibration_P', 'Calibration of the passive branch (class-balanced).'); print('\nCalibration of P:'); print(CALT.round(3).to_string(index=False))

sel = np.isin(SC, ['real_benign'] + [s for s in SCEN_ORDER if s.startswith('fake')]); ysel = (SC[sel] != 'real_benign').astype(int)
wsel = np.where(ysel == 1, .5 / ysel.sum(), .5 / (1 - ysel).sum())
def risk_coverage(s):
    o = np.argsort(s, kind='stable'); y = ysel[o]; w = wsel[o]; cov = np.cumsum(w) / w.sum(); risk = np.cumsum(w * y) / np.cumsum(w); return cov, risk
fig, ax = plt.subplots(figsize=(W1 + .3, 2.5)); aurc = {}
for key in ['W', 'P', 'R', 'AVG', 'B_WP', 'E_WP', 'F']:
    cov, rk = risk_coverage(TE[f's_{key}'].to_numpy()[sel]); aurc[key] = float((np.trapezoid if hasattr(np, "trapezoid") else np.trapz)(rk, cov)); ax.plot(cov, rk, color=COL[key], ls=LS[key], label=f'{MNAME[key]} (AURC={aurc[key]:.3f})')
ax.set_xlabel('coverage (fraction declared "real")'); ax.set_ylabel('risk (AI share among declared real)'); ax.set_title('Selective prediction'); ax.legend(fontsize=5.5)
savefig(fig, 'fig10_risk_coverage'); AURC = pd.DataFrame([dict(method=MNAME[k], AURC=v) for k, v in aurc.items()]); savetab(AURC, 'aurc', 'Area under the risk-coverage curve (lower is better).')

In [ ]:
# ============================================================================
# 14. DECISION STATES (taxonomy accuracy)  and  PRIOR SENSITIVITY of the FPR                  -> FIG 11a, FIG 12
# ============================================================================
STATES = ['AUTHENTIC_CONSISTENT', 'ABSTAIN', 'PROCESSED_UNVERIFIABLE', 'FORENSIC_AI', 'PROV_AI']
def decide(T, a, tau, kind='g'):
    """AI-origin evidence = Bonferroni(W,P) (valid FPR<=alpha on benign reals).  R is a PROCESSING indicator, not AI evidence:
    heavy processing without W/P evidence => 'PROCESSED_UNVERIFIABLE' (provenance cannot be verified), never 'AI'."""
    pAI, pW, pR = [T[f'p_{k}_{kind}'].to_numpy() for k in ('B_WP', 'W', 'R')]
    st = np.where(pAI > tau, 'AUTHENTIC_CONSISTENT', 'ABSTAIN').astype(object)
    st[pR <= a] = 'PROCESSED_UNVERIFIABLE'
    ai = pAI <= a; st[ai] = np.where(pW[ai] <= a / 2, 'PROV_AI', 'FORENSIC_AI'); return st
TE['state'] = decide(TE, ALPHA, CFG['TAU_ABSTAIN'])
sc_rows = ['real_benign', 'real_forged_wm', 'real_strong_edit', 'fake_unmarked_clean', 'fake_marked_clean', 'fake_marked_laundered', 'fake_unmarked_laundered']
CM = pd.crosstab(TE.scen, TE.state, normalize='index').reindex(index=sc_rows, columns=STATES, fill_value=0)
savetab(CM.reset_index(), 'decision_states', f'Decision-state distribution per scenario (alpha={ALPHA}, tau={CFG["TAU_ABSTAIN"]}).')
fig, ax = plt.subplots(figsize=(W1 + 1.8, 2.8)); im = ax.imshow(CM.values, cmap='Blues', vmin=0, vmax=1, aspect='auto')
for i in range(CM.shape[0]):
    for j in range(CM.shape[1]): ax.text(j, i, f'{CM.values[i, j]:.2f}', ha='center', va='center', fontsize=6, color='w' if CM.values[i, j] > .55 else 'k')
ax.set_xticks(range(len(STATES))); ax.set_xticklabels([s.replace('_', '\n') for s in STATES], fontsize=5.2); ax.set_yticks(range(len(sc_rows))); ax.set_yticklabels(sc_rows, fontsize=6); ax.grid(False)
ax.set_title('Decision states per scenario'); savefig(fig, 'fig12_decision_states'); print('\nDecision-state matrix:'); print(CM.round(2).to_string())

fb = {k: flag(TE, kk, kd, ALPHA)[SC == 'real_benign'].mean() for k, kk, kd in METH}; fs = {k: flag(TE, kk, kd, ALPHA)[SC == 'real_strong_edit'].mean() for k, kk, kd in METH}
pis = np.linspace(0, 1, 21); fig, ax = plt.subplots(figsize=(W1 + .3, 2.5))
for (name, _, _) in [m for m in METH if m[0] in ('P', 'R', 'B_WP', 'E_WP', 'E_WPR', 'F')]:
    ax.plot(pis, (1 - pis) * fb[name] + pis * fs[name], color=COL[name], ls=LS[name], label=MNAME[name])
ax.axvline(CFG['P_ADV_REAL'], color='gray', ls=':', lw=.7); ax.axhline(ALPHA, color='k', ls=':', lw=.7)
ax.set_xlabel('share $\\pi$ of heavily edited (tier-1, unmarked) images among real images'); ax.set_ylabel(f'real-image FPR @ $\\alpha$={ALPHA}')
ax.set_title('Sensitivity to the threat-model prior'); ax.legend(fontsize=5.5); savefig(fig, 'fig13_prior_sensitivity')

In [ ]:
# ============================================================================
# 14a. TAMPER-CHECK OPERATING PROFILE (question B, provider-marked content)   -> FIG 17
#      Detection of marked images by W / R / evidence-sum(W,P) / Bonferroni(W,P,R), and the COST: real images flagged.
# ============================================================================
TC_M = ['W', 'R', 'E_WP', 'BONF']; TC_S = [('fake_marked_clean', 'marked,\nclean'), ('fake_marked_benign', 'marked,\nbenign proc.'), ('fake_marked_laundered', 'marked,\nlaundered'),
                                            ('real_benign', 'real, benign\n(FPR)'), ('real_strong_edit', 'real, strong edit\n(cost)')]
dd_ = RES[(RES.alpha == ALPHA) & (RES.group == 'all')]; fig, ax = plt.subplots(figsize=(W2, 2.5)); wd = .8 / len(TC_M); tc_rows = []
for j, m_ in enumerate(TC_M):
    v = [dd_[(dd_.method == m_) & (dd_.scenario == s_)].iloc[0] for s_, _ in TC_S]
    ax.bar(np.arange(len(TC_S)) + (j - (len(TC_M) - 1) / 2) * wd, [x.rate for x in v], wd, yerr=[[x.rate - x.lo for x in v], [x.hi - x.rate for x in v]], color=COL[m_], ec='k', lw=.3,
           capsize=1, error_kw=dict(lw=.5), label=MNAME[m_])
    tc_rows += [dict(method=MNAME[m_], scenario=s_, rate=x.rate, lo=x.lo, hi=x.hi) for (s_, _), x in zip(TC_S, v)]
ax.set_xticks(range(len(TC_S))); ax.set_xticklabels([l for _, l in TC_S], fontsize=6); ax.set_ylabel(f'flag rate @ $\\alpha$={ALPHA}'); ax.axvline(2.5, color='k', lw=.6, ls=':')
ax.set_title('Provenance tamper check: detection of marked images vs. cost on real images'); ax.legend(ncol=4, fontsize=6, loc='upper center', bbox_to_anchor=(.5, -.28)); savefig(fig, 'fig17_tamper_check_profile')
savetab(pd.DataFrame(tc_rows), 'tamper_check_profile', f'Tamper-check operating profile at alpha={ALPHA} (all test views).')

In [ ]:
# ============================================================================
# 14b. MATCHED-PROCESSING EVALUATION  (v2)   -> FIG 16
#      Fake vs real images that received IDENTICAL processing (same watermark state, same attack cell).  Unmatched evaluation lets a
#      "was this image heavily processed?" signal masquerade as AI detection.  Pooling over cells is matched because every test image
#      is rendered under every cell.  mark=1 negatives are reals carrying a FORGED watermark.
# ============================================================================
GRP = np.where(TE.fam.to_numpy() == 'none', 'clean', np.where(TE.tier.to_numpy() == 0, 'benign', 'adversarial'))
LBL = TE.label.to_numpy(); MRK = TE.mark.to_numpy()
def auc_ci(pos, neg, score, B):
    m = pos | neg; img = TE.img.to_numpy()[m]; y = pos[m].astype(int); s_ = score[m]
    if y.min() == y.max(): return np.nan, np.nan, np.nan
    est = roc_auc_score(y, s_); u, inv = np.unique(img, return_inverse=True); by = [np.flatnonzero(inv == i) for i in range(len(u))]; bs = []
    for _ in range(B):
        rows = np.concatenate([by[i] for i in RNG_B.integers(0, len(u), len(u))]); yy = y[rows]
        if yy.min() != yy.max(): bs.append(roc_auc_score(yy, s_[rows]))
    return est, (np.quantile(bs, .025) if bs else np.nan), (np.quantile(bs, .975) if bs else np.nan)
MKEYS = ['W', 'P', 'R', 'B_WP', 'E_WP', 'E_WPR', 'F']; Bm = max(30, CFG['N_BOOT'] // 5); mrows = []
for mk in (0, 1):
    for g_ in ('clean', 'benign', 'adversarial'):
        pos = (LBL == 1) & (MRK == mk) & (GRP == g_); neg = (LBL == 0) & (MRK == mk) & (GRP == g_)
        for key in MKEYS:
            e_, lo_, hi_ = auc_ci(pos, neg, TE[f's_{key}'].to_numpy(), Bm)
            mrows.append(dict(watermark='marked (negatives = forged reals)' if mk else 'unmarked', processing=g_, method=MNAME[key], AUC=e_, lo=lo_, hi=hi_,
                              n_pos_img=TE.img[pos].nunique(), n_neg_img=TE.img[neg].nunique()))
MATCH = pd.DataFrame(mrows); savetab(MATCH, 'matched_processing_auc', 'AUC of fake vs real under identical processing (cluster-bootstrap 95% CI).')
fig, axs = plt.subplots(1, 2, figsize=(W2, 2.6), sharey=True)
for ax_, (wm_, ttl) in zip(axs, (('unmarked', '(a) unmarked: pure AI-origin task'), ('marked (negatives = forged reals)', '(b) marked vs FORGED-watermark reals'))):
    d = MATCH[MATCH.watermark == wm_]; w = .8 / len(MKEYS)
    for j, key in enumerate(MKEYS):
        dd = d[d.method == MNAME[key]].set_index('processing').reindex(['clean', 'benign', 'adversarial'])
        ax_.bar(np.arange(3) + (j - (len(MKEYS) - 1) / 2) * w, dd.AUC, w, yerr=[dd.AUC - dd.lo, dd.hi - dd.AUC], color=COL[key], ec='k', lw=.3, capsize=1, error_kw=dict(lw=.5), label=MNAME[key] if ax_ is axs[0] else None)
    ax_.axhline(.5, color='k', ls=':', lw=.8); ax_.set_xticks(range(3)); ax_.set_xticklabels(['clean', 'benign tier', 'adversarial tier']); ax_.set_ylim(0, 1.05); ax_.set_title(ttl)
axs[0].set_ylabel('AUC, fake vs real (matched processing)'); _h, _l = axs[0].get_legend_handles_labels(); fig.legend(_h, _l, loc='upper center', ncol=4, fontsize=6, bbox_to_anchor=(0.5, 1.12))
savefig(fig, 'fig16_matched_processing')
print('\nMatched-processing AUC (pooled over attack cells):'); print(MATCH.pivot_table(index='method', columns=['watermark', 'processing'], values='AUC').round(3).to_string())
mr = []
for mk in (0, 1):
    for g_ in ('clean', 'benign', 'adversarial'):
        pos = (LBL == 1) & (MRK == mk) & (GRP == g_); neg = (LBL == 0) & (MRK == mk) & (GRP == g_)
        for (name, key, kind) in METH:
            fl = flag(TE, key, kind, ALPHA); t_ = rate_ci(TE, pos, fl, 100); f_ = rate_ci(TE, neg, fl, 100)
            mr.append(dict(marked=mk, processing=g_, method=name, TPR=t_[0], TPR_lo=t_[1], TPR_hi=t_[2], FPR=f_[0], FPR_lo=f_[1], FPR_hi=f_[2]))
savetab(pd.DataFrame(mr), 'matched_processing_rates', f'TPR (fake) and FPR (real) under identical processing at alpha={ALPHA}.')

In [ ]:
# ============================================================================
# 14c. LEAVE-ONE-REAL-SOURCE-OUT CALIBRATION  (v2)   -> FIG 15
#      Recalibrate WITHOUT source s and measure FPR on source s.  Pooled conformal assumes the test reals are exchangeable with the
#      calibration pool; minimax (max over per-source p-values) is the proposed source-robust alternative.
# ============================================================================
cal_src = CAL['meta']['real_src'].to_numpy(); cal_sc = PL.cal_scores; mg_ = max(10, int(np.ceil(1 / min(CFG['ALPHAS']))) - 1)
all_srcs = sorted(TE[SC == 'real_benign'].real_src.unique()); lrows = []
for hs in all_srcs:
    keep = cal_src != hs; msk = (SC == 'real_benign') & (TE.real_src.to_numpy() == hs)
    if msk.sum() == 0 or keep.sum() < 20: continue
    ch = {k: Conformal(cal_sc[k][keep], cal_src[keep], mg_) for k in ('W', 'P', 'R', 'E_WP')}
    for kind_name, mm in (('pooled', False), ('minimax', True)):
        p = {k: ch[k].p(TE[f's_{k}'].to_numpy()[msk], minimax=mm) for k in ch}; p['B_WP'] = np.minimum(1, 2 * np.minimum(p['W'], p['P']))
        for key in ('W', 'P', 'R', 'B_WP', 'E_WP'):
            if key == 'B_WP' and mm: continue          # infeasible (see 'feasibility')
            fl = np.zeros(len(TE), bool); fl[msk] = p[key] <= ALPHA; e_ = rate_ci(TE, msk, fl, 100)
            lrows.append(dict(held_out_source=hs, never_in_calibration=hs in HOLD_REAL, calibration=kind_name, method=MNAME[key], FPR=e_[0], lo=e_[1], hi=e_[2], n_img=TE.img[msk].nunique()))
LORSO = pd.DataFrame(lrows); savetab(LORSO, 'lorso_fpr', f'Leave-one-real-source-out FPR at alpha={ALPHA}.')
WS = LORSO.groupby(['method', 'calibration']).FPR.max().unstack(); print(f'\nWorst held-out-source FPR at alpha={ALPHA} (nominal {ALPHA}):'); print(WS.round(3).to_string())
fig, ax = plt.subplots(figsize=(W2, 2.5)); series = [('Passive (P)', 'pooled', COL['P'], None), ('Evidence-sum(W,P)', 'pooled', COL['E_WP'], None),
                                                     ('Evidence-sum(W,P)', 'minimax', COL['E_WP'], '//')]
srcs_l = [s_ for s_ in all_srcs if s_ in set(LORSO.held_out_source)]; wd = .8 / len(series)
for j, (mn, ck, c_, h_) in enumerate(series):
    d = LORSO[(LORSO.method == mn) & (LORSO.calibration == ck)].set_index('held_out_source').reindex(srcs_l)
    ax.bar(np.arange(len(srcs_l)) + (j - (len(series) - 1) / 2) * wd, d.FPR, wd, yerr=[d.FPR - d.lo, d.hi - d.FPR], color=c_, hatch=h_, ec='k', lw=.3, capsize=1, error_kw=dict(lw=.5), label=f'{mn}, {ck}')
ax.axhline(ALPHA, color='k', ls=':', lw=.8); ax.set_xticks(range(len(srcs_l))); ax.set_xticklabels([s_ + ('*' if s_ in HOLD_REAL else '') for s_ in srcs_l], rotation=30, ha='right')
ax.set_ylabel(f'FPR on held-out source @ $\\alpha$={ALPHA}'); ax.set_title('Leave-one-real-source-out calibration (* never in calibration)'); ax.legend(fontsize=5.5); savefig(fig, 'fig15_lorso_fpr')

In [ ]:
# ============================================================================
# 15. EFFICIENCY: per-branch latency and cascade (W first; run P,R only if W does not resolve)   -> FIG 14
# ============================================================================
rs = np.random.default_rng(0); ib = rs.choice(len(TE_VT['zw']), size=min(96, len(TE_VT['zw'])), replace=False)
imgs_b = np.stack([apply_chain(IMGS[TE.img.iloc[i]], (), 0) for i in ib])
def timeit(fn, n):
    t = time.perf_counter(); fn(); return (time.perf_counter() - t) / n * 1000
c_W = timeit(lambda: [wm_decode(x) for x in imgs_b], len(imgs_b)); c_hf = timeit(lambda: [hf_features(x) for x in imgs_b], len(imgs_b))
ENC.encode(imgs_b[:8]); c_P = timeit(lambda: ENC.encode(imgs_b), len(imgs_b))
c_R = c_hf + 0.05; c_all = c_W + c_P + c_R
lat = []
for sc_ in ['fake_marked_clean', 'fake_marked_laundered', 'fake_unmarked_clean', 'real_benign']:
    msk = SC == sc_
    if msk.sum() == 0: continue
    resolved = flag(TE, 'W', 'g', ALPHA)[msk].mean(); lat.append(dict(scenario=sc_, resolved_by_W=resolved, parallel_ms=c_all, cascade_ms=c_W + (1 - resolved) * (c_P + c_R)))
LAT = pd.DataFrame(lat); savetab(LAT, 'latency_cascade', f'Per-image latency (ms): W={c_W:.1f}, P={c_P:.1f} ({ENC.name}), R={c_R:.1f}.')
fig, ax = plt.subplots(figsize=(W1 + .6, 2.4)); xi = np.arange(len(LAT))
ax.bar(xi - .18, LAT.parallel_ms, .36, color='#999999', ec='k', lw=.4, label='all branches'); ax.bar(xi + .18, LAT.cascade_ms, .36, color='#0072B2', ec='k', lw=.4, label='cascade (W first)')
ax.set_xticks(xi); ax.set_xticklabels([s.replace('_', '\n', 1) for s in LAT.scenario], fontsize=6); ax.set_ylabel('ms / image'); ax.set_title('Cost of the cascade'); ax.legend(); savefig(fig, 'fig14_latency')
print('\nLatency (ms/img): W=%.1f  hf=%.1f  P(encoder)=%.1f' % (c_W, c_hf, c_P)); print(LAT.round(2).to_string(index=False))

In [ ]:
# ============================================================================
# 16. TRAINING-SEED VARIANCE (refit P, R, fusion; 80% image subsample for seeds > 0) and FUSION ABLATION
# ============================================================================
KEY_SC = ['real_benign', 'fake_unmarked_clean', 'fake_marked_laundered', 'fake_unmarked_laundered', 'real_forged_wm', 'real_strong_edit']
seed_rows = []
for sd in range(CFG['N_SEEDS']):
    pls = PL if sd == 0 else Pipeline(TR, FU, CAL, seed=sd, subsample=0.8); Ts = TE if sd == 0 else make_TE(pls, TE_VT); scs = Ts['scen'].to_numpy()
    for (name, key, kind) in METH:
        for s_ in KEY_SC: seed_rows.append(dict(seed=sd, method=name, scenario=s_, rate=flag(Ts, key, kind, ALPHA)[scs == s_].mean()))
SEEDS = pd.DataFrame(seed_rows); SS = SEEDS.groupby(['method', 'scenario']).rate.agg(['mean', 'std']).reset_index()
SS['mean+-sd'] = SS.apply(lambda r: f"{r['mean']:.3f}+-{(0 if np.isnan(r['std']) else r['std']):.3f}", axis=1)
SST = SS.pivot(index='method', columns='scenario', values='mean+-sd').reindex([m[0] for m in METH])[KEY_SC]; savetab(SST.reset_index(), 'seed_variance', f'Mean+-sd over {CFG["N_SEEDS"]} training seeds at alpha={ALPHA}.')
print(f'\nTraining-seed variance (alpha={ALPHA}, {CFG["N_SEEDS"]} seeds)'); print(SST.to_string())
abl = []
for name in ['W', 'P', 'R', 'B_WP', 'E_WP', 'BONF', 'E_WPR', 'F_WP', 'F_WR', 'F_PR', 'F']:
    r = dict(model={'F_WP': 'LR W+P', 'F_WR': 'LR W+R', 'F_PR': 'LR P+R', 'F': 'LR W+P+R (v1)', 'B_WP': 'Bonferroni W+P', 'BONF': 'Bonferroni W+P+R',
                    'E_WP': 'Evidence-sum W+P', 'E_WPR': 'Evidence-sum W+P+R'}.get(name, name + ' only'))
    for s_ in ['fake_unmarked_clean', 'fake_marked_clean', 'fake_marked_laundered', 'fake_unmarked_laundered']: r['TPR ' + s_] = flag(TE, name, 'g', ALPHA)[SC == s_].mean()
    for s_ in ['real_benign', 'real_forged_wm', 'real_strong_edit']: r['FPR ' + s_] = flag(TE, name, 'g', ALPHA)[SC == s_].mean()
    abl.append(r)
ABL = pd.DataFrame(abl); savetab(ABL, 'ablation_branches', f'Branch ablation at alpha={ALPHA}.'); print('\nBranch ablation:'); print(ABL.round(3).to_string(index=False))

## Adaptive attacker, quality constraint, hardening (v2)
* **Black-box adaptive attacker (oracle scores):** for each provider-marked AI image it draws `K` random laundering chains (1-2 operations,
  *continuous* strengths, all five families incl. the two unseen in training), queries the detector it targets and keeps the best candidate.
* **Quality-constrained attacker:** the same attacker may only keep candidates whose PSNR w.r.t. the marked image is at least `q` dB; if none qualifies
  it must submit the un-attacked image. ASR is therefore reported against image quality, not as a bare number.
* **Hardening round:** the hardest candidates against the *deployed* detector (`CFG['HARD_TARGET']`), drawn only from training families, are added to
  P/R/fusion training; everything is refit and **re-calibrated** on the same held-out real images.
* **White-box PGD** against the passive head, with a *veto test*: does a learned fusion still see the watermark after P is attacked?
* Guarantee scope: conformal validity covers un-attacked, benign real images only.

In [ ]:
# ============================================================================
# 17. ADAPTIVE ATTACKER vs ROUND-0 AND HARDENED MODELS (with quality constraint)               -> FIG 11
# ============================================================================
rng = np.random.default_rng(CFG['SEED'] + 9)
test_fake_seen = MAN.index[(MAN.split == 'test') & (MAN.label == 1) & (~MAN.hold)].to_numpy()
adv_ids = rng.choice(test_fake_seen, size=min(CFG['N_ADV'], len(test_fake_seen)), replace=False); NADV = len(adv_ids)
K = CFG['K_MAX']; KS = [k for k in (1, 2, 4, 8, 16, 24, 32) if k <= K]
CAND = compute_views(candidate_specs_cont(adv_ids, K, ALL_FAMS, rng, mark=1), ENC, want_psnr=True, desc='adaptive candidates')
ADAPT_T = ['W', 'P', 'R', 'F', 'B_WP', 'BONF', 'E_WP', 'E_WPR']; QS = CFG['PSNR_FRONTIER']

_rci = np.random.default_rng(CFG['SEED'] + 77)
def _boot_ci(b, B=200):
    b = np.asarray(b, float); m = b[_rci.integers(0, len(b), (B, len(b)))].mean(1); return float(np.quantile(m, .025)), float(np.quantile(m, .975))

def adaptive_eval(plx, tag):
    sc = plx.scores(CAND); pv = plx.pvalues(sc); Tx = make_TE(plx, TE_VT); n = NADV; ar = np.arange(n); PS = CAND['psnr'].reshape(n, K); rows = []
    msk = (Tx.scen.to_numpy() == 'fake_marked_clean') & np.isin(Tx.img.to_numpy(), adv_ids)
    for t in ADAPT_T:
        S = (-pv[f'p_{t}_g'] if t in ('B_WP', 'BONF') else sc[t]).reshape(n, K); Pm = pv[f'p_{t}_g'].reshape(n, K)
        base = pd.Series(Tx[f'p_{t}_g'].to_numpy()[msk], index=Tx.img.to_numpy()[msk]).groupby(level=0).first().reindex(adv_ids).to_numpy(); base_ev = base > ALPHA
        lo_, hi_ = _boot_ci(base_ev); rows.append(dict(model=tag, target=t, min_psnr='none', K=0, ASR=float(np.mean(base_ev)), ASR_lo=lo_, ASR_hi=hi_, psnr_dB=np.nan))
        for q in QS:
            for k in KS:
                feas = np.ones((n, k), bool) if q is None else (PS[:, :k] >= q)
                j = np.argmin(np.where(feas, S[:, :k], np.inf), axis=1); has = feas.any(1); ev = np.where(has, Pm[ar, j] > ALPHA, base_ev)
                ps_sel = np.where(has, PS[ar, j], np.nan)
                lo_, hi_ = _boot_ci(ev); rows.append(dict(model=tag, target=t, min_psnr='none' if q is None else str(q), K=k, ASR=float(np.mean(ev)), ASR_lo=lo_, ASR_hi=hi_, psnr_dB=float(np.nanmedian(ps_sel)) if has.any() else np.nan))
    return pd.DataFrame(rows), Tx
AD0, TE0 = adaptive_eval(PL, 'round-0')
print(f'\nAdaptive attacker vs round-0 (n={NADV} images; K={KS[-1]}):'); print(AD0[(AD0.K == KS[-1]) & (AD0.min_psnr.isin(['none', '30', '35']))].pivot_table(index='target', columns='min_psnr', values='ASR').round(3).to_string())

def hard_views(plx, img_ids, Kh, fams, rng_):
    vt = compute_views(candidate_specs(img_ids, Kh, fams, rng_, mark=1), ENC, desc='hard-example mining'); n = len(img_ids)
    j = np.argmin(plx.scores(vt)[CFG['HARD_TARGET']].reshape(n, Kh), axis=1); return vt_take(vt, np.arange(n) * Kh + j)
PL1, TE1, AD1 = None, None, None
if CFG['RUN_HARDENING']:
    trf = MAN.index[(MAN.split == 'train') & (MAN.label == 1)].to_numpy(); fuf = MAN.index[(MAN.split == 'fusion') & (MAN.label == 1)].to_numpy()
    HT = hard_views(PL, rng.choice(trf, min(CFG['N_HARD_TR'], len(trf)), replace=False), CFG['K_HARD'], CFG['A_TRAIN'], rng)
    HFV = hard_views(PL, rng.choice(fuf, min(CFG['N_HARD_FU'], len(fuf)), replace=False), CFG['K_HARD'], CFG['A_TRAIN'], rng)
    PL1 = Pipeline(TR, FU, CAL, seed=0, extra_tr=HT, extra_fu=HFV); AD1, TE1 = adaptive_eval(PL1, 'hardened')
    hr = []
    for tag, Tx in (('round-0', TE0), ('hardened', TE1)):
        scx = Tx['scen'].to_numpy()
        for key in ('E_WP', 'B_WP', 'F'):
            for s_ in ['real_benign', 'real_strong_edit', 'fake_unmarked_clean', 'fake_marked_clean', 'fake_marked_laundered', 'fake_unmarked_laundered']:
                hr.append(dict(model=tag, detector=key, scenario=s_, rate=float(flag(Tx, key, 'g', ALPHA)[scx == s_].mean())))
    HARD = pd.DataFrame(hr); savetab(HARD, 'hardening_effect', f'Flag rates before/after one hardening round (alpha={ALPHA}).')
    print('\nHardening effect:'); print(HARD.pivot_table(index=['detector', 'scenario'], columns='model', values='rate').round(3).to_string())
AD = pd.concat([AD0] + ([AD1] if AD1 is not None else []), ignore_index=True); savetab(AD, 'adaptive_attack_asr', f'Adaptive black-box attack success vs query budget K and minimum PSNR (alpha={ALPHA}).')

fig, axs = plt.subplots(1, 3, figsize=(W2, 2.6)); show = ['W', 'P', 'F', 'B_WP', 'E_WP', 'E_WPR']
for t in show:
    d = AD0[(AD0.target == t) & (AD0.min_psnr == 'none')]; axs[0].plot(d.K.replace(0, .7), d.ASR, color=COL[t], ls=LS[t], marker=MK[t], ms=3, label=MNAME[t])
axs[0].set_xscale('log', base=2); axs[0].set_xticks([.7] + KS); axs[0].set_xticklabels(['none'] + [str(k) for k in KS]); axs[0].minorticks_off()
axs[0].set_xlabel('query budget $K$'); axs[0].set_ylabel(f'attack success @ $\\alpha$={ALPHA}'); axs[0].set_title('(a) round-0, no quality limit'); axs[0].set_ylim(-.02, 1.02)
qlab = ['none'] + [str(q) for q in QS if q is not None]
for tag, ls_ in (('round-0', '-'), ('hardened', '--')):
    if tag == 'hardened' and AD1 is None: continue
    for t in ('F', 'B_WP', 'E_WP', 'E_WPR'):
        d = AD[(AD.model == tag) & (AD.target == t) & (AD.K == KS[-1])].set_index('min_psnr').reindex(qlab)
        axs[1].plot(range(len(qlab)), d.ASR, color=COL[t], ls=ls_, marker=MK[t], ms=3)
axs[1].set_xticks(range(len(qlab))); axs[1].set_xticklabels(qlab); axs[1].set_xlabel(f'minimum PSNR of attacked image (dB), $K$={KS[-1]}'); axs[1].set_title('(b) success vs quality'); axs[1].set_ylim(-.02, 1.02)
fin = AD[(AD.K == KS[-1]) & (AD.min_psnr == 'none')]; tg = ['W', 'P', 'R', 'F', 'B_WP', 'E_WP', 'E_WPR']; xi = np.arange(len(tg)); mods = list(fin.model.unique())
for j, tag in enumerate(mods):
    v = [fin[(fin.model == tag) & (fin.target == t)].ASR.iloc[0] for t in tg]; axs[2].bar(xi + (j - .5 * (len(mods) - 1)) * .36, v, .36, ec='k', lw=.4, label=tag, color=['#999999', '#0072B2'][j])
axs[2].set_xticks(xi); axs[2].set_xticklabels(tg, fontsize=6, rotation=30); axs[2].set_title(f'(c) ASR at $K$={KS[-1]}'); axs[2].legend(); axs[2].set_ylim(0, 1.02)
_h, _l = axs[0].get_legend_handles_labels(); fig.legend(_h, _l, loc='upper center', ncol=6, fontsize=6, bbox_to_anchor=(0.5, 1.10)); savefig(fig, 'fig11_adaptive_attack')

In [ ]:
# ============================================================================
# 18. WHITE-BOX PGD against the passive head (GPU encoder only)  +  VETO TEST
#     If ASR(fusion) >> ASR(W) after PGD, a branch that "looks real" is cancelling verified watermark evidence (the v1 failure mode).
# ============================================================================
PGDT = None; PGD_KEYS = ['W', 'P', 'R', 'F', 'B_WP', 'BONF', 'E_WP', 'E_WPR']
def pgd_attack_P(plx, imgs, eps255, steps, bs=32):
    torch = ENC.torch; dev = ENC.dev
    mu = torch.tensor(plx.P['scaler'].mean_, dtype=torch.float32, device=dev); sd = torch.tensor(plx.P['scaler'].scale_, dtype=torch.float32, device=dev)
    w = torch.tensor(plx.P['lr'].coef_[0], dtype=torch.float32, device=dev); b0 = float(plx.P['lr'].intercept_[0]); out = []
    for i in range(0, len(imgs), bs):
        x0 = torch.from_numpy(imgs[i:i + bs]).to(dev).permute(0, 3, 1, 2).float() / 255.; eps = eps255 / 255.; a = eps / 4; x = x0.clone()
        for _ in range(steps):
            x.requires_grad_(True)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=dev == 'cuda'): e = ENC.forward_t(x)
            logit = ((e.float() - mu) / sd) @ w + b0
            g, = torch.autograd.grad(logit.sum(), x)
            x = x.detach() - a * g.sign(); x = torch.min(torch.max(x, x0 - eps), x0 + eps).clamp(0, 1)
        out.append((x.detach() * 255).round().clamp(0, 255).byte().permute(0, 2, 3, 1).cpu().numpy())
    return np.concatenate(out)
if CFG['RUN_PGD'] and hasattr(ENC, 'forward_t'):
    try:
        pg_ids = adv_ids[:CFG['N_PGD']]; x0 = np.stack([wm_embed(IMGS[i]) for i in pg_ids]); xb = np.stack([atk_blur(x, 2.5) for x in x0]); res = []
        for variant, base_imgs in (('PGD only', x0), ('blur(2.5) then PGD', xb)):
            for eps in CFG['PGD_EPS']:
                for tag, plx in (('round-0', PL), ('hardened', PL1)):
                    if plx is None: continue
                    xa = pgd_attack_P(plx, base_imgs, eps, CFG['PGD_STEPS']); ft = featurize_images(xa, ENC)
                    vt = dict(ft, meta=pd.DataFrame({'bin': MAN['bin'].to_numpy()[pg_ids]}), psnr=np.full(len(xa), np.nan))
                    sc = plx.scores(vt); pv = plx.pvalues(sc)
                    res.append(dict(variant=variant, eps_255=eps, model=tag, psnr_vs_input_dB=np.mean([psnr(a, b) for a, b in zip(base_imgs, xa)]), psnr_vs_marked_dB=np.mean([psnr(a, b) for a, b in zip(x0, xa)]), **{f'ASR_{t}': float(np.mean(pv[f'p_{t}_g'] > ALPHA)) for t in PGD_KEYS}))
        PGDT = pd.DataFrame(res); savetab(PGDT, 'pgd_whitebox', f'White-box PGD vs passive head (alpha={ALPHA}); ASR = not flagged.'); print('\nWhite-box PGD (veto test: compare ASR_F / ASR_E_WP with ASR_W):'); print(PGDT.round(3).to_string(index=False))
    except Exception as e:
        log('PGD section failed (non-fatal):', repr(e))
else:
    log('PGD skipped (smoke mode or CPU-only encoder).')

if PGDT is not None:
    pm_ = PGDT[(PGDT.model == 'round-0') & (PGDT.variant == 'PGD only') & (PGDT.eps_255 == PGDT.eps_255.max())].iloc[0]
    dff = RES[(RES.alpha == ALPHA) & (RES.group == 'all') & (RES.scenario == 'real_forged_wm')].set_index('method').rate
    fig, ax = plt.subplots(figsize=(W1 + .9, 2.8))
    for k in ['W', 'P', 'F', 'B_WP', 'E_WP', 'BONF', 'E_WPR']:
        ax.scatter(dff[k], pm_[f'ASR_{k}'], color=COL[k], marker=MK[k], s=45, ec='k', lw=.4, label=MNAME[k])
    ax.set_xlabel('FPR on forged-watermark reals'); ax.set_ylabel(f'PGD attack success ($\\epsilon$={pm_.eps_255}/255)')
    ax.set_xlim(-.03, 1.03); ax.set_ylim(-.03, 1.03); ax.set_title('Veto vs. forgery trade-off (lower is better on both axes)', fontsize=7)
    ax.legend(fontsize=5.5, ncol=2, loc='upper center', bbox_to_anchor=(.5, -.2)); savefig(fig, 'fig18_veto_vs_forgery_tradeoff')

In [ ]:
# ============================================================================
# 19. EXPORT + AUTOMATIC REVIEWER CHECKLIST (v2)
# ============================================================================
import shutil
ind = (SC == 'real_benign') & ~UN; CHK = []
def chk(name, ok, detail): CHK.append(dict(check=name, status='PASS' if ok else 'WARN', detail=detail))
uninformative = MAN[['fmt', 'sizetier']].drop_duplicates().shape[0] == 1
chk('metadata leakage probe AUC < 0.75', LEAK_AUC < .75, f'AUC={LEAK_AUC:.3f}' + (' (probe is UNINFORMATIVE here: all images share format and size; content/domain confounding is NOT tested)' if uninformative else ''))
chk('calibration size supports alpha', len(CAL['meta']) >= 1 / ALPHA - 1, f"n_cal={len(CAL['meta'])}, need >= {1/ALPHA-1:.0f} (smallest reportable alpha = {1/(len(CAL['meta'])+1):.4f})")
for _nm in MAINS + ['E_WP-mm']:
    _f = FEAS[(FEAS.method == _nm) & (FEAS.alpha == ALPHA)].iloc[0]; chk(f'detector can flag at alpha ({MNAME[_nm]})', _f.feasible, f'min attainable p = {_f.min_attainable_p:.4f} vs alpha={ALPHA}')
for key in MAINS:
    e = rate_ci(TE, ind, flag(TE, key, 'g', ALPHA)); chk(f'in-distribution FPR <= alpha ({MNAME[key]})', e[1] <= ALPHA, f'FPR={e[0]:.3f} [{e[1]:.3f},{e[2]:.3f}] vs alpha={ALPHA}')
for (nm, key, kd) in [m for m in METH if m[0] in ('E_WP', 'B_WP', 'E_WP-mm', 'B_WP-mm', 'P')]:
    w_ = SRCT[SRCT.method == nm]; worst = w_.loc[w_.FPR.idxmax()]
    chk(f'worst-source FPR <= 2 alpha ({MNAME[nm]})', worst.FPR <= 2 * ALPHA, f'{worst.FPR:.3f} on {worst.source} [{worst.lo:.3f},{worst.hi:.3f}]')
mu_ = MATCH[(MATCH.watermark == 'unmarked') & (MATCH.processing == 'clean') & (MATCH.method.isin([MNAME['P'], MNAME['E_WP'], MNAME['B_WP']]))]
chk('AI-origin AUC under matched processing >= 0.70 (unmarked, clean)', mu_.AUC.max() >= .70, 'best = ' + ', '.join(f'{r.method}: {r.AUC:.3f}' for r in mu_.itertuples()))
ma = MATCH[(MATCH.watermark == 'unmarked') & (MATCH.processing == 'adversarial')]
chk('AI-origin AUC under matched ADVERSARIAL processing >= 0.60', ma.AUC.max() >= .60, 'best = ' + ', '.join(f'{r.method}: {r.AUC:.3f}' for r in ma.itertuples() if r.method in (MNAME['P'], MNAME['E_WP'], MNAME['R'])))
for s_ in ['fake_marked_clean', 'fake_marked_laundered']:
    msk = SC == s_
    if msk.sum() == 0: continue
    d = diff_ci(TE, msk, flag(TE, 'E_WP', 'g', ALPHA), flag(TE, 'F', 'g', ALPHA)); chk(f'evidence-sum(W,P) beats v1 LR fusion on {s_}', d[1] > 0, f'diff TPR = {d[0]:+.3f} [{d[1]:+.3f},{d[2]:+.3f}]')
d = diff_ci(TE, SC == 'real_forged_wm', flag(TE, 'W', 'g', ALPHA), flag(TE, 'E_WP', 'g', ALPHA)); chk('adding P reduces forged-watermark false alarms vs W-only', d[1] > 0, f'FPR(W)-FPR(E_WP) = {d[0]:+.3f} [{d[1]:+.3f},{d[2]:+.3f}]')
if PGDT is not None:
    pm = PGDT[(PGDT.model == 'round-0') & (PGDT.variant == 'PGD only') & (PGDT.eps_255 == PGDT.eps_255.max())].iloc[0]
    chk('no veto under PGD: ASR(E_WP) <= ASR(W) + 0.05', pm.ASR_E_WP <= pm.ASR_W + .05, f'eps={pm.eps_255}: ASR_W={pm.ASR_W:.3f}, ASR_E_WP={pm.ASR_E_WP:.3f}, ASR_F(v1)={pm.ASR_F:.3f}')
for t in ('E_WP', 'E_WPR'):
    a0 = AD0[(AD0.target == t) & (AD0.K == KS[-1]) & (AD0.min_psnr == 'none')].ASR.iloc[0]; a35 = AD0[(AD0.target == t) & (AD0.K == KS[-1]) & (AD0.min_psnr == '35')].ASR.iloc[0]
    chk(f'adaptive ASR ({MNAME[t]}, K={KS[-1]}) reported with quality constraint', True, f'no limit {a0:.3f} -> PSNR>=35 dB {a35:.3f} (n={NADV} images)')
if AD1 is not None:
    t = CFG['HARD_TARGET']; a0 = AD0[(AD0.target == t) & (AD0.K == KS[-1]) & (AD0.min_psnr == 'none')].ASR.iloc[0]; a1 = AD1[(AD1.target == t) & (AD1.K == KS[-1]) & (AD1.min_psnr == 'none')].ASR.iloc[0]
    fb0 = flag(TE0, t, 'g', ALPHA)[(TE0.scen.to_numpy() == 'real_benign')].mean(); fb1 = flag(TE1, t, 'g', ALPHA)[(TE1.scen.to_numpy() == 'real_benign')].mean()
    chk(f'hardening lowers adaptive ASR without raising real-image FPR ({MNAME[t]})', (a1 < a0) and (fb1 <= fb0 + .01), f'ASR {a0:.3f}->{a1:.3f}; FPR(real_benign) {fb0:.3f}->{fb1:.3f}')
CHKT = pd.DataFrame(CHK); savetab(CHKT, 'reviewer_checklist', 'Automatic sanity checks before writing claims.'); print('\n' + '=' * 30, 'REVIEWER CHECKLIST', '=' * 30); print(CHKT.to_string(index=False))

fpr_m = {k: rate_ci(TE, ind, flag(TE, k, 'g', ALPHA)) for k in MAINS}
summary = dict(version='v2', preset=PRESET, alpha_main=ALPHA, n_images=int(len(MAN)), n_cal_reals=int(len(CAL['meta'])), leakage_auc=float(LEAK_AUC), wm_alpha=CFG['WM_ALPHA'], wm_psnr_db=psnr_wm,
               fpr_in_distribution={k: dict(zip(['est', 'lo', 'hi'], map(float, v))) for k, v in fpr_m.items()}, worst_source_fpr=WORST.to_dict(),
               encoder=ENC.name, holdout_generators=HOLD_FAKE, holdout_real_source=HOLD_REAL, runtime_s=time.time() - T0)
(OUT / 'summary.json').write_text(json.dumps(summary, indent=2, default=str)); (OUT / 'config.json').write_text(json.dumps(CFG, indent=2, default=str))
env = dict(python=sys.version.split()[0], platform=platform.platform(), numpy=np.__version__, pandas=pd.__version__, matplotlib=matplotlib.__version__)
import sklearn, scipy; env.update(sklearn=sklearn.__version__, scipy=scipy.__version__)
try:
    import torch, transformers; env.update(torch=torch.__version__, transformers=transformers.__version__, cuda=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
except Exception: pass
(OUT / 'environment.json').write_text(json.dumps(env, indent=2)); MAN.drop(columns=['path']).to_csv(OUT / 'manifest_no_paths.csv', index=False)
shutil.make_archive(str(OUT.parent / 'results_bundle'), 'zip', str(OUT)); log('DONE. Results bundle:', OUT.parent / 'results_bundle.zip')